# GPU-Based K-Means Clustering Using CUDA — Problem 5

Lloyd's K-Means on an NVIDIA GPU in CUDA C++: given $N$ points in $\mathbb{R}^D$ and $K$ initial
centroids, alternate between assigning every point to its nearest centroid by squared Euclidean
distance and recomputing every centroid as the mean of its assigned points, until assignments stop
changing or the largest centroid movement is $\le \varepsilon$, or until the iteration limit is
reached. The program reports final assignments, final centroids, the iteration count and the
execution time.

The two centroid-accumulation strategies required by the assignment — **Approach A (global
atomics)** and **Approach B (deterministic two-stage reduction)** — are both implemented and
selectable at run time, and the assignment kernel exists in a baseline and a shared-memory variant.

> **This notebook is generated.** `tools/build_notebook.py` assembles it from the files under
> `src/`, `tests/` and `tools/cells/`, and its `%%writefile` cells are checked to be byte-identical
> to those files (DEC-011). Never edit the notebook by hand: change the repository files and
> regenerate, or the next regeneration silently discards the edit.

## Design decisions (requirements.md §3.1)

| Decision | Selected policy |
|---|---|
| Centroid accumulation (A) | Element-wise global `atomicAdd` on binary64 sums and `uint32` counts (§6) |
| Centroid accumulation (B) | Two-stage ownership-based reduction: chunk-privatised partial sums, then a fixed-order tree combine (DEC-001) — race-free without atomics, and deterministic |
| Convergence | Assignment stability **or** $\Delta \le \varepsilon$ (Euclidean, inclusive); stability takes reporting precedence (DEC-002) |
| Empty clusters | Retain the previous centroid; excluded from $\Delta$ (DEC-003) |
| Final consistency | Report the last $(A^{(t)}, C^{(t)})$ pair; P1 holds by construction, P2 verified and reported as a flag (DEC-004) |
| Tie-breaking | Lowest centroid index on exact binary32 equality, via strict-less-than replacement in ascending $k$ (DEC-005) |
| Iteration counting | Number of completed centroid updates (DEC-006) |
| Numeric types | binary32 points, centroids and distances; binary64 centroid sums, movement and tolerance; `uint32` counts (DEC-007) |
| Assignment mapping | One thread per point, grid-stride, ascending-$j$ distance accumulation (DEC-008) |
| Device layout | Two read-only copies of $X$: dimension-major for assignment and Approach A, row-major for Approach B Stage 1 (DEC-009) |
| Global synchronisation | Kernel boundaries on a single stream only; no cooperative-groups grid sync (DEC-010) |
| Deliverable | This single notebook, generated from `src/` (DEC-011) |
| Workflow | Edit and test locally, push to GitHub, run on a Colab GPU, save the executed notebook under `results/` (DEC-012) |

## How to run

1. **Runtime → Change runtime type → GPU** (ENV-11). The configuration cell fails fast with no GPU.
2. Run the cells top to bottom. The environment cell records the machine (§1.6.2), the
   configuration cell derives `SM_ARCH` from the recorded compute capability, and the smoke test
   proves the toolchain before any program source is compiled.
3. Save the executed notebook with **File → Save a copy in GitHub** to
   `GPU-Based-K-Means-Clustering/results/kmeans_cuda_<YYYYMMDD-HHMM>.ipynb` (DEC-012). GPU results
   count as evidence only from an executed notebook committed there.

## 2. Environment record

Records the machine-dependent items left open by requirements.md §1.6.2 — ENV-01 (GPU model and
compute capability), ENV-02 (CUDA toolkit, driver, `compute-sanitizer`), ENV-03 (host OS and host
compiler) and ENV-08 (profiler availability) — into the `ENV` dictionary that every later cell
reads, and that PERF-008 requires in each benchmark CSV row.

No version is assumed anywhere: whatever `nvidia-smi`, `nvcc` and `g++` report is what is recorded
and printed. A missing tool is recorded as unavailable rather than raising, because ENV-08 allows
the profilers to be absent (PERF-009 and TEST-P03 then fall back to CUDA-event phase timing, which
is reported as a limitation). The configuration cell, not this one, decides whether the recorded
environment is usable.

`nvidia-smi` does not report the SM count, and ENV-06 (FP64:FP32 throughput ratio) is a vendor
specification figure, not a queryable property. The SM count comes from
`cudaGetDeviceProperties` — printed by the smoke test below and required of the program itself by
CUDA-001 — and ENV-06 is cited from the vendor specification in the evaluation section.

In [ ]:
# Environment record — requirements.md §1.6.2 (ENV-01..ENV-03, ENV-08), PERF-008.
# Every probe is tolerant: a missing tool is recorded as unavailable, never raised. Nothing here
# assumes a GPU model, a toolkit version or a driver version; the reported strings are the record.
import json
import platform
import re
import shutil
import subprocess
import sys


def _run(cmd, timeout=120):
    """Run a probe command. Returns (ok, text); never raises."""
    if shutil.which(cmd[0]) is None:
        return False, "not installed"
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception as exc:  # a probe must not break the notebook
        return False, f"probe failed: {exc!r}"
    out = (p.stdout or "").strip() or (p.stderr or "").strip()
    return p.returncode == 0, out


_VERSION_RE = re.compile(r"[0-9]+[.][0-9]+")


def _version_of(cmd):
    """Version line of a tool invoked as `<tool> --version`, or None.

    compute-sanitizer and ncu print the product name on the first line and the version several
    lines later, so taking the first line alone recorded no version at all for them (ENV-02,
    ENV-08). Prefer the first line that carries a version number, keeping the product name with
    it, and fall back to the first line when no line has one.
    """
    ok, out = _run([cmd, "--version"])
    if not ok:
        return None
    lines = [line.strip() for line in out.splitlines() if line.strip()]
    if not lines:
        return None
    if _VERSION_RE.search(lines[0]):
        return lines[0]
    for line in lines[1:]:
        if _VERSION_RE.search(line) and "opyright" not in line:
            return f"{lines[0]} {line}"
    return lines[0]


ENV = {}

# ENV-03 — host OS and host compiler of the runtime VM.
ENV["os"] = {
    "platform": platform.platform(),
    "release": _run(["cat", "/etc/os-release"])[1].replace("\n", "; ") or None,
    "python": sys.version.split()[0],
}
ENV["host_compiler"] = _version_of("g++")

# ENV-01 — GPU model and compute capability.
gpu_ok, gpu_csv = _run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total",
     "--format=csv,noheader"]
)
ENV["gpu"] = None
if gpu_ok and gpu_csv:
    fields = [f.strip() for f in gpu_csv.splitlines()[0].split(",")]
    if len(fields) == 4:
        ENV["gpu"] = {
            "name": fields[0],
            "compute_cap": fields[1],
            "driver_version": fields[2],
            "memory_total": fields[3],
        }
        ENV["gpu_count"] = len(gpu_csv.splitlines())
if ENV["gpu"] is None:
    ENV["gpu_probe"] = gpu_csv or "nvidia-smi unavailable"

# ENV-02 — CUDA toolkit and the sanitizer the §16.4 suite needs.
nvcc_ok, nvcc_out = _run(["nvcc", "--version"])
ENV["nvcc"] = nvcc_out.replace("\n", " | ") if nvcc_ok else None
ENV["compute_sanitizer"] = _version_of("compute-sanitizer")

# ENV-08 — profiler availability; absence is a recorded limitation, not an error.
ENV["profilers"] = {"nsys": _version_of("nsys"), "ncu": _version_of("ncu")}

# Notebook analysis stack (§1.5 deliverables 4 and 5; no other dependency is permitted).
ENV["python_packages"] = {}
for _pkg in ("numpy", "pandas", "matplotlib"):
    try:
        ENV["python_packages"][_pkg] = __import__(_pkg).__version__
    except Exception:
        ENV["python_packages"][_pkg] = None

print(json.dumps(ENV, indent=2))

## 3. Configuration

Derives everything machine-dependent from the record above — nothing is hard-coded before ENV-01
is known (BLD-001c):

- **`SM_ARCH`** from the reported compute capability (`7.5` → `sm_75`).
- **Fail fast** when the environment cannot run the assignment: no GPU attached, compute capability
  below 6.0 (CUDA-001 — `atomicAdd` on `double`, which ATOM-003 requires, is not native below
  6.0), or a CUDA 13.x toolkit paired with a device below CC 7.5, whose offline compilation that
  toolkit removed (BLD-001a). Each case raises with the action to take, so "restart kernel and run
  all" stops here rather than failing later inside a build.
- **Build flags** (BLD-002): a release build `-O3 -lineinfo -arch=$SM_ARCH -std=c++17` with
  `-Xptxas -v` so register and shared-memory usage appear in the build log (CUDA-010), and a
  sanitizer/debug build that adds `-DKMEANS_SYNC_CHECK` (CUDA-008). `--use_fast_math` is never
  used (NUM-002), and the CPU reference is compiled with FP contraction disabled (NUM-003).
- **Paths**: results — benchmark CSVs, output files and logs — go to Google Drive when
  `USE_DRIVE` is enabled and the mount succeeds, so a runtime disconnect does not lose them;
  otherwise a warning is printed, local disk is used, and the fallback is recorded in `ENV` so
  results that were not persisted stay identifiable (BLD-001c). Sources, binaries and generated
  datasets always stay on local disk: the Drive FUSE mount does not permit executing binaries, so
  anything built there could not be run, and datasets are reproducible from their seeds (FR-003)
  while reaching tens of GB at the top of the PERF-003 matrix. The cell `chdir`s into the build
  directory, which is why the `%%writefile` cells use flat file names.

In [ ]:
# Configuration — requirements.md §15.3 part 3; BLD-001a, BLD-001c, BLD-002, CUDA-001, NUM-002/003.
import os
import re
import subprocess

USE_DRIVE = True  # BLD-001c: persist results to Drive; falls back to local disk on failure.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/kmeans_cuda"
BUILD_DIR = "/content/kmeans"  # always local: see the note on exec permissions below

if "ENV" not in globals():
    raise RuntimeError("Run the environment cell (section 2) first: ENV is not defined.")

# --- ENV-01 -> SM_ARCH (BLD-001c: derived, never hard-coded) ------------------------------------
if not ENV.get("gpu"):
    raise RuntimeError(
        "No NVIDIA GPU visible (ENV-01). Set Runtime -> Change runtime type -> Hardware "
        "accelerator: GPU, then re-run from section 2. Probe said: "
        f"{ENV.get('gpu_probe', 'unknown')}"
    )

COMPUTE_CAP = ENV["gpu"]["compute_cap"].strip()
_cc_match = re.fullmatch(r"(\d+)\.(\d+)", COMPUTE_CAP)
if not _cc_match:
    raise RuntimeError(f"Could not parse compute capability {COMPUTE_CAP!r} reported by nvidia-smi.")
CC_MAJOR, CC_MINOR = int(_cc_match.group(1)), int(_cc_match.group(2))
CC_VALUE = CC_MAJOR + CC_MINOR / 10.0
SM_ARCH = f"sm_{CC_MAJOR}{CC_MINOR}"

# CUDA-001: ATOM-003 needs native atomicAdd(double), which arrived with compute capability 6.0.
if CC_VALUE < 6.0:
    raise RuntimeError(
        f"Compute capability {COMPUTE_CAP} is below the 6.0 minimum (CUDA-001): native "
        "atomicAdd(double) required by ATOM-003 is unavailable. Use a newer GPU runtime."
    )

# ENV-02: whatever nvcc reports is what is used; no toolkit version is assumed.
NVCC_VERSION = ENV.get("nvcc")
if not NVCC_VERSION:
    raise RuntimeError("nvcc is not available in this runtime (ENV-02); CUDA code cannot be built.")
_rel = re.search(r"release (\d+)\.(\d+)", NVCC_VERSION)
CUDA_TOOLKIT = f"{_rel.group(1)}.{_rel.group(2)}" if _rel else "unknown"

# BLD-001a: CUDA 13.x dropped offline compilation for compute capability < 7.5.
if _rel and int(_rel.group(1)) >= 13 and CC_VALUE < 7.5:
    raise RuntimeError(
        f"CUDA toolkit {CUDA_TOOLKIT} cannot compile offline for compute capability "
        f"{COMPUTE_CAP} (BLD-001a). Use a CUDA 12.x toolkit, or a GPU with CC >= 7.5."
    )

# --- BLD-002 build flags ------------------------------------------------------------------------
# No --use_fast_math anywhere (NUM-002): it would change the distance semantics of §2.2.
NVCC_COMMON = ["-std=c++17", f"-arch={SM_ARCH}", "-lineinfo"]
NVCC_RELEASE_FLAGS = NVCC_COMMON + ["-O3", "-Xptxas", "-v"]          # CUDA-010 usage in build log
NVCC_DEBUG_FLAGS = NVCC_COMMON + ["-O3", "-DKMEANS_SYNC_CHECK"]      # CUDA-008 localises faults
# NUM-003: the CPU reference is the oracle and must not contract multiply-adds.
NVCC_HOST_FP_FLAGS = ["-Xcompiler", "-ffp-contract=off"]
GXX_REFERENCE_FLAGS = ["-O2", "-std=c++17", "-ffp-contract=off"]

# --- BLD-001c paths ----------------------------------------------------------------------------
# Sources, binaries and generated datasets stay on local disk and only results go to Drive, for
# two reasons: the Drive FUSE mount does not permit executing binaries, so a kmeans built there
# could not be run, and datasets reach tens of GB at the top of the PERF-003 matrix while being
# reproducible from their seeds (FR-003), so they are not worth persisting.
RESULTS_ROOT, DRIVE_STATUS = BUILD_DIR, "disabled"
if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        RESULTS_ROOT, DRIVE_STATUS = DRIVE_RESULTS_DIR, "mounted"
    except Exception as exc:
        DRIVE_STATUS = f"unavailable: {exc!r}"
        print(
            "warning[BLD-001c]: Google Drive not mounted; results stay in "
            f"{BUILD_DIR} and are lost if the runtime disconnects, so download anything you "
            f"need before closing the session. Reason: {exc!r}"
        )

WORK_DIR = BUILD_DIR                              # cwd for the %%writefile cells and the compiler
DATA_DIR = os.path.join(BUILD_DIR, "data")        # generated datasets (FR-003), regenerable
OUT_DIR = os.path.join(RESULTS_ROOT, "out")       # .kma/.kmd outputs and JSON summaries (BLD-004)
CSV_DIR = os.path.join(RESULTS_ROOT, "csv")       # benchmark CSVs (section 13.1)
LOG_DIR = os.path.join(RESULTS_ROOT, "logs")      # test and sanitizer logs (section 1.5)
for _d in (BUILD_DIR, DATA_DIR, OUT_DIR, CSV_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)
os.chdir(WORK_DIR)  # the %%writefile cells below use flat names relative to this directory

# PERF-008 asks for a build identity per benchmark row. In Colab there is no git checkout, so the
# executed notebook saved under results/ is itself the evidence (DEC-012); record what is knowable.
try:
    RUN_VERSION = "git:" + subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=30
    ).stdout.strip()
except Exception:
    RUN_VERSION = "unknown"
if RUN_VERSION == "git:":
    RUN_VERSION = "unknown"

ENV["drive_status"] = DRIVE_STATUS          # BLD-001c: unpersisted results stay identifiable
ENV["work_dir"] = WORK_DIR
ENV["results_root"] = RESULTS_ROOT
ENV["sm_arch"] = SM_ARCH
ENV["cuda_toolkit"] = CUDA_TOOLKIT
ENV["run_version"] = RUN_VERSION

print(f"GPU             : {ENV['gpu']['name']} (compute capability {COMPUTE_CAP})")
print(f"SM_ARCH         : {SM_ARCH}   [derived from ENV-01, BLD-001c]")
print(f"CUDA toolkit    : {CUDA_TOOLKIT} (as reported by nvcc)")
print(f"Drive           : {DRIVE_STATUS}")
print(f"build dir (cwd) : {WORK_DIR}   [local: Drive mounts are not executable]")
print(f"results root    : {RESULTS_ROOT}")
print(f"data / out / csv: {DATA_DIR} | {OUT_DIR} | {CSV_DIR}")
print(f"run version     : {RUN_VERSION}")
print(f"nvcc release    : {' '.join(NVCC_RELEASE_FLAGS)}")
print(f"nvcc debug      : {' '.join(NVCC_DEBUG_FLAGS)}")
print(f"host FP (oracle): {' '.join(NVCC_HOST_FP_FLAGS)} / g++ {' '.join(GXX_REFERENCE_FLAGS)}")

## 4. Toolchain smoke test

A nine-line kernel is cheaper to debug than a multi-file build, so before any program source is
compiled this section writes `hello.cu`, builds it with the **same release flags the real build
uses** (BLD-002, so a bad flag is caught here), runs it, and checks three things:

1. `nvcc` accepts `-arch=$SM_ARCH` for the attached device and the binary links and runs;
2. a kernel launches and its writes are visible to the host (`kernel.check=OK`, verified
   element-wise, not just by a checksum);
3. the compute capability the device reports to host code equals the one `nvidia-smi` reported in
   section 2 — if these disagree, `SM_ARCH` was derived from the wrong device.

The cell raises on a non-zero compile or run exit code, on a mismatch, or on a missing `OK`, so
"restart kernel and run all" halts at the first toolchain fault (NFR-014). It also prints the SM
count, shared memory per block, warp size and the CUDA runtime and driver versions — the
`cudaGetDeviceProperties` fields that `nvidia-smi` does not expose, which CUDA-001 requires the
program to query, MEM-004 and SMEM-003 check tile sizes against, and CUDA-007 uses to size grids.
Whatever the runtime reports is printed verbatim; no CUDA version is assumed.

In [ ]:
%%writefile hello.cu
// Toolchain smoke test (requirements.md §15.3 part 4).
//
// Proves, before any program source is compiled, that the SM_ARCH derived from ENV-01 matches the
// attached device, that the toolkit reported by nvcc can build and link for it, and that a kernel
// launches and its writes are visible to the host. It also prints the device properties that
// nvidia-smi does not expose (SM count, shared memory per block) and that CUDA-001 requires the
// program itself to query, plus the CUDA runtime and driver versions exactly as reported — no
// version is assumed anywhere.
//
// Output is `key=value` lines so the notebook cell can assert on them. Exit codes follow BLD-005:
// 0 success, 4 CUDA runtime error.

#include <cstdio>
#include <cstdlib>

#include <cuda_runtime.h>

// CUDA-008: every runtime call is checked; failure reports file, line, call and the error string.
#define CUDA_CHECK(call)                                                                     \
    do {                                                                                     \
        cudaError_t err_ = (call);                                                            \
        if (err_ != cudaSuccess) {                                                            \
            std::fprintf(stderr, "error[CUDA-008]: %s:%d: %s failed: %s\n", __FILE__,          \
                         __LINE__, #call, cudaGetErrorString(err_));                           \
            std::exit(4);                                                                      \
        }                                                                                     \
    } while (0)

static const int kElements = 256;

// One thread per element, grid-stride (CUDA-007), bounds-checked (CUDA-005). The value is i*i so
// the host can verify every element independently of launch geometry.
__global__ void square_indices(int *out, int n)
{
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x) {
        out[i] = i * i;
    }
}

int main()
{
    int device = 0;
    CUDA_CHECK(cudaGetDevice(&device));

    cudaDeviceProp prop{};
    CUDA_CHECK(cudaGetDeviceProperties(&prop, device));

    int runtime_version = 0;
    int driver_version = 0;
    CUDA_CHECK(cudaRuntimeGetVersion(&runtime_version));
    CUDA_CHECK(cudaDriverGetVersion(&driver_version));

    std::printf("device.name=%s\n", prop.name);
    std::printf("device.compute_capability=%d.%d\n", prop.major, prop.minor);
    std::printf("device.sm_count=%d\n", prop.multiProcessorCount);
    std::printf("device.global_mem_bytes=%zu\n", prop.totalGlobalMem);
    std::printf("device.shared_mem_per_block_bytes=%zu\n", prop.sharedMemPerBlock);
    std::printf("device.shared_mem_per_block_optin_bytes=%zu\n", prop.sharedMemPerBlockOptin);
    std::printf("device.warp_size=%d\n", prop.warpSize);
    std::printf("device.max_threads_per_block=%d\n", prop.maxThreadsPerBlock);
    std::printf("runtime.cuda_version=%d\n", runtime_version);
    std::printf("driver.cuda_version=%d\n", driver_version);

    int *d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, kElements * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_out, 0, kElements * sizeof(int)));

    square_indices<<<4, 64>>>(d_out, kElements);
    CUDA_CHECK(cudaGetLastError());  // launch error (CUDA-008)

    int host_out[kElements];
    CUDA_CHECK(cudaMemcpy(host_out, d_out, sizeof(host_out), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));

    long long sum = 0;
    long long expected_sum = 0;
    int mismatches = 0;
    for (int i = 0; i < kElements; ++i) {
        sum += host_out[i];
        expected_sum += static_cast<long long>(i) * i;
        if (host_out[i] != i * i) {
            ++mismatches;
        }
    }

    std::printf("kernel.elements=%d\n", kElements);
    std::printf("kernel.sum=%lld\n", sum);
    std::printf("kernel.expected_sum=%lld\n", expected_sum);
    std::printf("kernel.mismatches=%d\n", mismatches);
    std::printf("kernel.check=%s\n", (mismatches == 0 && sum == expected_sum) ? "OK" : "FAIL");

    return (mismatches == 0 && sum == expected_sum) ? 0 : 1;
}


In [ ]:
# Toolchain smoke test — requirements.md §15.3 part 4. Raises on any failure so that
# "restart kernel and run all" stops here instead of inside a larger build (NFR-014).
import subprocess

if "SM_ARCH" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first: SM_ARCH is not defined.")

compile_cmd = ["nvcc"] + NVCC_RELEASE_FLAGS + ["-o", "hello", "hello.cu"]
print("$ " + " ".join(compile_cmd))
compile_proc = subprocess.run(compile_cmd, capture_output=True, text=True)
print(compile_proc.stdout + compile_proc.stderr, end="")  # -Xptxas -v usage report (CUDA-010)
if compile_proc.returncode != 0:
    raise RuntimeError(
        f"nvcc failed with exit code {compile_proc.returncode} for -arch={SM_ARCH} "
        f"(toolkit {CUDA_TOOLKIT}, device CC {COMPUTE_CAP}). See the output above; check BLD-001a "
        "if the toolkit is too new for this device."
    )

run_proc = subprocess.run(["./hello"], capture_output=True, text=True)
print(run_proc.stdout + run_proc.stderr, end="")
if run_proc.returncode != 0:
    raise RuntimeError(
        f"./hello exited with code {run_proc.returncode} "
        "(4 = CUDA runtime error per BLD-005). See the output above."
    )

smoke = dict(
    line.split("=", 1) for line in run_proc.stdout.splitlines() if "=" in line
)
if smoke.get("kernel.check") != "OK":
    raise RuntimeError(
        f"Kernel result wrong: {smoke.get('kernel.mismatches')} of "
        f"{smoke.get('kernel.elements')} elements mismatched. The toolchain builds but the device "
        "does not compute correctly; do not trust any later result."
    )
if smoke.get("device.compute_capability") != COMPUTE_CAP:
    raise RuntimeError(
        f"Device reports compute capability {smoke.get('device.compute_capability')} to host code "
        f"but nvidia-smi reported {COMPUTE_CAP} in section 2, so SM_ARCH={SM_ARCH} was derived "
        "from a different device (BLD-001c). Re-run from section 2."
    )

# SM count feeds CUDA-007 grid sizing and the Approach B chunk count G (RED-006); the shared-memory
# limits bound the assign_smem tiles (SMEM-003, MEM-004). Recorded now, used from M2 on.
ENV["device_properties"] = smoke
SM_COUNT = int(smoke["device.sm_count"])
SHARED_MEM_PER_BLOCK = int(smoke["device.shared_mem_per_block_bytes"])

print(
    f"\nsmoke test OK: {smoke['device.name']}, CC {smoke['device.compute_capability']}, "
    f"{SM_COUNT} SMs, {SHARED_MEM_PER_BLOCK} B shared memory per block, "
    f"CUDA runtime {smoke['runtime.cuda_version']}, driver {smoke['driver.cuda_version']}, "
    f"kernel sum {smoke['kernel.sum']} == expected {smoke['kernel.expected_sum']}"
)

## 5. Source files

Each cell below writes one repository file verbatim into the working directory. They are generated
from `src/` and `tests/` by `tools/build_notebook.py`, which also checks that what the notebook
writes is byte-identical to what the repository holds (DEC-011) — so editing a cell here is always
wrong: change the file in the repository and regenerate.

This milestone (M1) covers the host side only, in the ARC order of §18.2:

| File | Component | Responsibility |
|---|---|---|
| `common.h` | ARC-01 | `Status` (§2.7), exit codes (BLD-005), message format (BLD-006), the NUM-005/NUM-006 tolerances, and overflow-checked size arithmetic with the MEM-002 capacity model (CUDA-006) |
| `validate.h/.cpp` | ARC-01 | VAL-001…VAL-010, including the VAL-007 range bound $4DM^2 \le 2^{126}$ that guarantees no binary32 distance can overflow during the run |
| `io_kmd.h/.cpp` | ARC-02 | the `.kmd`/`.kma` binary formats of BLD-004, with the FR-002 exact-size check |
| `cpu_reference.h/.cpp` | ARC-03 | the oracle of NUM-003: the FP32 distance path of §2.2, a binary64 path for near-tie margins, binary64 sums, the tie rule, empty-cluster retention, the convergence loop and finalization. The step functions are exposed individually for the §10.3 lockstep verification added in M5 |
| `gen.h/.cpp`, `gen_main.cpp` | ARC-02 | the FR-003 generator and the `kmeans-gen` CLI of BLD-003a |
| `hand_data.h`, `test_cpu.cpp` | ARC-12 | H1–H6 and H2-a…e with their §16.0 expected outcomes, plus TEST-U08, U09, U11, U12 and U13 at CPU level |

The CUDA kernels, the device context and the `kmeans` driver are not here yet: they arrive in M2
once this reference passes, because every GPU result is judged against it.

In [ ]:
%%writefile common.h
// common.h — shared types, statuses, exit codes, messages and size arithmetic.
//
// requirements.md: Status (§2.7), ExitCode (BLD-005), message format (BLD-006),
// overflow-checked size arithmetic (CUDA-006) and the capacity model (MEM-002).
//
// Header-only and free of CUDA constructs, so the M2 .cu translation units include it unchanged.

#ifndef KMEANS_COMMON_H
#define KMEANS_COMMON_H

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <string>
#include <vector>

namespace kmeans {

// --- §2.7 termination statuses -----------------------------------------------------------------

enum class Status {
    CONVERGED_ASSIGNMENTS,  // A(t) == A(t-1) exactly
    CONVERGED_MOVEMENT,     // delta <= epsilon, assignments still changing
    ITERATION_LIMIT,        // t == max_iter and neither criterion held (includes max_iter == 0)
    ERROR                   // validation, resource, CUDA or I/O failure; no clustering output
};

inline const char *status_name(Status s)
{
    switch (s) {
    case Status::CONVERGED_ASSIGNMENTS: return "CONVERGED_ASSIGNMENTS";
    case Status::CONVERGED_MOVEMENT:    return "CONVERGED_MOVEMENT";
    case Status::ITERATION_LIMIT:       return "ITERATION_LIMIT";
    case Status::ERROR:                 return "ERROR";
    }
    return "UNKNOWN";
}

// §2.7: termination is never reported as convergence.
inline bool is_converged(Status s)
{
    return s == Status::CONVERGED_ASSIGNMENTS || s == Status::CONVERGED_MOVEMENT;
}

// --- BLD-005 exit codes ------------------------------------------------------------------------

enum ExitCode {
    kExitOk = 0,                  // any non-error status
    kExitInvalidInput = 2,        // VAL rule violated
    kExitUnsupportedDevice = 3,   // unsupported device or insufficient resources
    kExitCudaError = 4,           // CUDA runtime error
    kExitIoError = 5,             // I/O error
    kExitVerificationFailed = 6   // only with --verify
};

// --- BLD-006 messages --------------------------------------------------------------------------
// Single-line prefix naming the rule that was violated, then the cause and the offending value.

inline void report_error(const char *rule_id, const std::string &what)
{
    std::fprintf(stderr, "error[%s]: %s\n", rule_id, what.c_str());
}

inline void report_warning(const char *warning_id, const std::string &what)
{
    std::fprintf(stderr, "warning[%s]: %s\n", warning_id, what.c_str());
}

// --- numerical constants (§0.1, NUM-005, NUM-006) ----------------------------------------------

// u32 = 2^-24, the unit roundoff of IEEE-754 binary32.
constexpr double kUnitRoundoffF32 = 5.9604644775390625e-08;

// NUM-005: a point is a near-tie when its relative margin is at most 4 (D + 2) u32.
inline double near_tie_threshold(int64_t D)
{
    return 4.0 * static_cast<double>(D + 2) * kUnitRoundoffF32;
}

// NUM-006: |a - b| <= 4 u32 |b| + 4 u32 S_max 2^-10, with a the value under test and b the
// reference. Both sides round one binary64 quotient to binary32, so they differ by at most about
// one binary32 ulp; the absolute term covers values near zero.
inline double centroid_tolerance(float reference, double s_max)
{
    const double abs_ref = std::fabs(static_cast<double>(reference));
    return 4.0 * kUnitRoundoffF32 * abs_ref +
           4.0 * kUnitRoundoffF32 * s_max * 0.0009765625;  // 2^-10
}

inline bool centroid_agrees(float test, float reference, double s_max)
{
    const double diff = std::fabs(static_cast<double>(test) - static_cast<double>(reference));
    return diff <= centroid_tolerance(reference, s_max);
}

// NUM-006: S_max = max(M, 1e-30), with M the range bound of VAL-007.
inline double s_max_from_range(double m_range)
{
    return m_range > 1e-30 ? m_range : 1e-30;
}

// --- CUDA-006 / MEM-002 size arithmetic --------------------------------------------------------
// Every element offset and allocation size is computed in unsigned 64-bit arithmetic with an
// explicit overflow check, so an oversized configuration is rejected before any allocation
// instead of wrapping into a small, plausible-looking byte count.

inline bool checked_mul(uint64_t a, uint64_t b, uint64_t *out)
{
    if (a != 0 && b > UINT64_MAX / a) {
        return false;
    }
    *out = a * b;
    return true;
}

inline bool checked_add(uint64_t a, uint64_t b, uint64_t *out)
{
    if (a > UINT64_MAX - b) {
        return false;
    }
    *out = a + b;
    return true;
}

// Byte counts of MEM-001, in the order of that table. `reduce_chunks` is the Approach B chunk
// count G; pass 0 for Approach A, whose run allocates no P/Q buffers.
struct MemoryModel {
    uint64_t x_col = 0;        // 4ND
    uint64_t x_row = 0;        // 4ND
    uint64_t centroids = 0;    // 8KD   (C_cur and C_next)
    uint64_t assignments = 0;  // 4N
    uint64_t sums = 0;         // 8KD
    uint64_t counts = 0;       // 4K
    uint64_t partials = 0;     // 8GKD
    uint64_t partial_counts = 0;  // 4GK
    uint64_t scalars = 0;      // movement block maxima, delta, change counter, pinned scalars
    uint64_t total = 0;
    bool overflow = false;     // true when any product or sum exceeded 64 bits
};

// MEM-002: required bytes = 8ND + 4N + 16KD + 4K + [8GKD + 4GK] + small.
inline MemoryModel device_memory_bytes(int64_t N, int64_t D, int64_t K, int64_t reduce_chunks,
                                       int64_t movement_blocks = 1024)
{
    MemoryModel m;
    const uint64_t n = static_cast<uint64_t>(N);
    const uint64_t d = static_cast<uint64_t>(D);
    const uint64_t k = static_cast<uint64_t>(K);
    const uint64_t g = static_cast<uint64_t>(reduce_chunks);

    uint64_t nd = 0, kd = 0, gkd = 0, gk = 0;
    bool ok = checked_mul(n, d, &nd) && checked_mul(k, d, &kd);
    ok = ok && checked_mul(g, kd, &gkd) && checked_mul(g, k, &gk);
    if (!ok) {
        m.overflow = true;
        return m;
    }

    ok = ok && checked_mul(nd, 4, &m.x_col);
    ok = ok && checked_mul(nd, 4, &m.x_row);
    ok = ok && checked_mul(kd, 8, &m.centroids);
    ok = ok && checked_mul(n, 4, &m.assignments);
    ok = ok && checked_mul(kd, 8, &m.sums);
    ok = ok && checked_mul(k, 4, &m.counts);
    ok = ok && checked_mul(gkd, 8, &m.partials);
    ok = ok && checked_mul(gk, 4, &m.partial_counts);
    ok = ok && checked_mul(static_cast<uint64_t>(movement_blocks), 8, &m.scalars);
    ok = ok && checked_add(m.scalars, 32, &m.scalars);  // delta, change counter, pinned copies
    if (!ok) {
        m.overflow = true;
        return m;
    }

    const uint64_t parts[] = {m.x_col,  m.x_row, m.centroids, m.assignments,  m.sums,
                              m.counts, m.partials, m.partial_counts, m.scalars};
    uint64_t total = 0;
    for (uint64_t part : parts) {
        if (!checked_add(total, part, &total)) {
            m.overflow = true;
            return m;
        }
    }
    m.total = total;
    return m;
}

// --- run configuration and result --------------------------------------------------------------

enum class Strategy { kAtomic, kReduction };
enum class AssignKernel { kBaseline, kSharedMemory };

struct Config {
    int64_t N = 0;
    int64_t D = 0;
    int64_t K = 0;
    int64_t max_iter = 0;
    double tol = 0.0;
    bool fixed_iters = false;  // FR-020: disable both convergence criteria
    Strategy strategy = Strategy::kAtomic;
    AssignKernel assign_kernel = AssignKernel::kBaseline;
    int block_size = 256;          // CUDA-007 default
    int64_t reduce_chunks = 0;     // 0 = choose automatically (RED-006)
};

// §2.8 step 3 reports exactly these fields; the CPU reference and the device driver fill the same
// record so the two can be compared field by field (§10.3).
struct Result {
    std::vector<int32_t> assignments;  // A_final
    std::vector<float> centroids;      // C_final, K x D row-major, unchanged by finalization
    std::vector<uint32_t> counts;      // n_final from the finalization accumulation pass
    int64_t iterations = 0;            // DEC-006: completed centroid updates
    Status status = Status::ERROR;
    double last_delta = 0.0;           // delta of the last update
    bool has_last_delta = false;       // false when t == 0 (reported as null, FR-015)
    bool centroid_consistent = false;  // P2, verified per NUM-006
    bool assignment_consistent = true; // P1, true by construction (§2.8)
    double objective = 0.0;            // J (§2.9), binary64, outside timed regions
};

}  // namespace kmeans

#endif  // KMEANS_COMMON_H


In [ ]:
%%writefile validate.h
// validate.h — input validation rules VAL-001..VAL-010 (requirements.md §9.1, FR-001).
//
// Every rule is applied before any device allocation, and the first violation names the parameter
// and the rule ID (BLD-006) and maps to an exit code (BLD-005). Rules that accept the input with a
// warning (VAL-003 K > N, VAL-008 duplicate centroids) record the warning ID instead of failing.

#ifndef KMEANS_VALIDATE_H
#define KMEANS_VALIDATE_H

#include <string>
#include <vector>

#include "common.h"

namespace kmeans {

struct Validation {
    bool ok = true;
    std::string rule;     // failing rule ID, empty when ok
    std::string message;  // cause and offending value
    int exit_code = kExitOk;
    std::vector<std::string> warnings;  // e.g. "W-KGTN", "W-DUPC", in the order detected

    bool has_warning(const std::string &id) const;
    // Emit the failure and every warning in BLD-006 format.
    void report() const;
};

// VAL-001, VAL-002, VAL-003, VAL-004, VAL-005. Sizes arrive as 64-bit integers (§2.1) and are
// rejected unless they fit the int32 assignment and uint32 count types.
Validation validate_parameters(int64_t N, int64_t D, int64_t K, int64_t max_iter, double tol);

// VAL-010: option values within their documented ranges and the device limits. Device limits are
// passed in so the rule is testable without a GPU; pass 0 for a limit that is not yet known.
Validation validate_options(int block_size, int64_t reduce_chunks, int tile_k, int tile_d,
                            int device_max_threads_per_block, int64_t device_shared_mem_per_block);

// VAL-007 range bound: M = max(max|x_ij|, max|c0_kj|). Returns M, which NUM-006 also needs as the
// scale S_max. Returns a negative value only if the arrays are empty.
double range_bound(const std::vector<float> &X, const std::vector<float> &C0);

// VAL-006 (finite values, first offending row and column reported), VAL-007 (4 D M^2 <= 2^126
// evaluated in binary64) and VAL-008 (shape of the centroid file, duplicate centroids warned).
// `m_range` receives the M of VAL-007 when validation succeeds.
Validation validate_data(const std::vector<float> &X, int64_t N, int64_t D,
                         const std::vector<float> &C0, int64_t K, double *m_range);

// VAL-009: total required device memory (MEM-002) must fit in the free device memory less a 5 %
// safety margin. `free_bytes` comes from cudaMemGetInfo in M2 and from the test in M1.
Validation validate_capacity(int64_t N, int64_t D, int64_t K, int64_t reduce_chunks,
                             uint64_t free_bytes);

}  // namespace kmeans

#endif  // KMEANS_VALIDATE_H


In [ ]:
%%writefile validate.cpp
#include "validate.h"

#include <algorithm>
#include <cmath>
#include <map>

namespace kmeans {
namespace {

constexpr int64_t kInt32Max = 2147483647;  // 2^31 - 1 (VAL-002, VAL-004)

Validation fail(const char *rule, const std::string &message, int exit_code)
{
    Validation v;
    v.ok = false;
    v.rule = rule;
    v.message = message;
    v.exit_code = exit_code;
    return v;
}

// Distinct names rather than overloads: an overloaded num(int) would be ambiguous between the integral and
// floating-point conversions, and every message below states exactly which kind of value it holds.
std::string istr(int64_t value)
{
    return std::to_string(value);
}

std::string ustr(uint64_t value)
{
    return std::to_string(value);
}

std::string dstr(double value)
{
    char buf[64];
    std::snprintf(buf, sizeof(buf), "%.17g", value);
    return std::string(buf);
}

}  // namespace

bool Validation::has_warning(const std::string &id) const
{
    return std::find(warnings.begin(), warnings.end(), id) != warnings.end();
}

void Validation::report() const
{
    for (const std::string &w : warnings) {
        if (w == "W-KGTN") {
            report_warning(w.c_str(), "K > N: at least K - N clusters stay empty and retain their "
                                      "initial centroids (VAL-003, DEC-003)");
        } else if (w == "W-DUPC") {
            report_warning(w.c_str(), "duplicate initial centroids: the higher-index duplicate "
                                      "receives no points while identical (VAL-008, DEC-005)");
        } else {
            report_warning(w.c_str(), "see requirements.md");
        }
    }
    if (!ok) {
        report_error(rule.c_str(), message);
    }
}

Validation validate_parameters(int64_t N, int64_t D, int64_t K, int64_t max_iter, double tol)
{
    // VAL-001
    if (N < 1) {
        return fail("VAL-001", "N must be at least 1, got N=" + istr(N), kExitInvalidInput);
    }
    if (D < 1) {
        return fail("VAL-001", "D must be at least 1, got D=" + istr(D), kExitInvalidInput);
    }
    if (K < 1) {
        return fail("VAL-001", "K must be at least 1, got K=" + istr(K), kExitInvalidInput);
    }

    // VAL-002: assignments must fit int32 and counts uint32.
    if (N > kInt32Max) {
        return fail("VAL-002", "N must be at most 2^31-1, got N=" + istr(N), kExitInvalidInput);
    }
    if (D > kInt32Max) {
        return fail("VAL-002", "D must be at most 2^31-1, got D=" + istr(D), kExitInvalidInput);
    }
    if (K > kInt32Max) {
        return fail("VAL-002", "K must be at most 2^31-1, got K=" + istr(K), kExitInvalidInput);
    }

    // VAL-004: max_iter = 0 is accepted and means "initial assignment only" (§2.7).
    if (max_iter < 0 || max_iter > kInt32Max) {
        return fail("VAL-004", "max-iter must be in [0, 2^31-1], got " + istr(max_iter),
                    kExitInvalidInput);
    }

    // VAL-005: epsilon finite and non-negative; NaN, +-Inf and negative values are rejected.
    if (std::isnan(tol)) {
        return fail("VAL-005", "tol must be a finite number, got NaN", kExitInvalidInput);
    }
    if (std::isinf(tol)) {
        return fail("VAL-005", "tol must be finite, got " + dstr(tol), kExitInvalidInput);
    }
    if (tol < 0.0) {
        return fail("VAL-005", "tol must be non-negative, got " + dstr(tol), kExitInvalidInput);
    }

    Validation v;
    // VAL-003: K > N is accepted with a warning; at least K - N clusters remain empty (DEC-003).
    if (K > N) {
        v.warnings.push_back("W-KGTN");
    }
    return v;
}

Validation validate_options(int block_size, int64_t reduce_chunks, int tile_k, int tile_d,
                            int device_max_threads_per_block, int64_t device_shared_mem_per_block)
{
    // CUDA-007: block size is a multiple of 32 in [32, 1024] and within the device limit.
    if (block_size < 32 || block_size > 1024 || block_size % 32 != 0) {
        return fail("VAL-010",
                    "block-size must be a multiple of 32 in [32, 1024], got " + istr(block_size),
                    kExitInvalidInput);
    }
    if (device_max_threads_per_block > 0 && block_size > device_max_threads_per_block) {
        return fail("VAL-010", "block-size " + istr(block_size) + " exceeds the device limit of " +
                                   istr(device_max_threads_per_block),
                    kExitInvalidInput);
    }

    // RED-006: G is chosen automatically when 0, otherwise it must be at least 1.
    if (reduce_chunks < 0) {
        return fail("VAL-010", "reduce-chunks must be at least 1 (or 0 for automatic), got " +
                                   istr(reduce_chunks),
                    kExitInvalidInput);
    }

    // SMEM-001/SMEM-003: tile sizes positive, T_D a multiple of the warp size, 4 T_K T_D bytes of
    // shared memory within the per-block limit (EDGE-34 rejects oversized configurations).
    if (tile_k < 1 || tile_d < 1) {
        return fail("VAL-010", "tile sizes must be at least 1, got tile_k=" + istr(tile_k) +
                                   " tile_d=" + istr(tile_d),
                    kExitInvalidInput);
    }
    if (tile_d % 32 != 0) {
        return fail("VAL-010", "tile_d must be a multiple of the warp size 32, got " + istr(tile_d),
                    kExitInvalidInput);
    }
    uint64_t shared_bytes = 0;
    if (!checked_mul(static_cast<uint64_t>(tile_k) * 4, static_cast<uint64_t>(tile_d),
                     &shared_bytes)) {
        return fail("VAL-010", "tile sizes overflow the shared-memory calculation",
                    kExitInvalidInput);
    }
    if (device_shared_mem_per_block > 0 &&
        shared_bytes > static_cast<uint64_t>(device_shared_mem_per_block)) {
        return fail("VAL-010", "assign_smem tiles need " + ustr(shared_bytes) +
                                   " B of shared memory, device allows " +
                                   istr(device_shared_mem_per_block) + " B per block",
                    kExitInvalidInput);
    }
    return Validation{};
}

double range_bound(const std::vector<float> &X, const std::vector<float> &C0)
{
    double m = -1.0;
    for (float value : X) {
        const double a = std::fabs(static_cast<double>(value));
        if (a > m) {
            m = a;
        }
    }
    for (float value : C0) {
        const double a = std::fabs(static_cast<double>(value));
        if (a > m) {
            m = a;
        }
    }
    return m;
}

Validation validate_data(const std::vector<float> &X, int64_t N, int64_t D,
                         const std::vector<float> &C0, int64_t K, double *m_range)
{
    Validation v;

    // VAL-008: the centroid file has exactly K rows of D values; the data file N rows of D.
    const uint64_t expect_x = static_cast<uint64_t>(N) * static_cast<uint64_t>(D);
    const uint64_t expect_c = static_cast<uint64_t>(K) * static_cast<uint64_t>(D);
    if (X.size() != expect_x) {
        return fail("VAL-008", "data has " + ustr(X.size()) +
                                   " values, expected N*D = " + ustr(expect_x),
                    kExitInvalidInput);
    }
    if (C0.size() != expect_c) {
        return fail("VAL-008", "centroids have " + ustr(C0.size()) +
                                   " values, expected K*D = " + ustr(expect_c),
                    kExitInvalidInput);
    }

    // VAL-006: every coordinate finite, reporting the first offending (row, column).
    for (int64_t i = 0; i < N; ++i) {
        for (int64_t j = 0; j < D; ++j) {
            const float value = X[static_cast<size_t>(i * D + j)];
            if (!std::isfinite(value)) {
                return fail("VAL-006", "data value at (row " + istr(i) + ", column " + istr(j) +
                                           ") is not finite: " + dstr(static_cast<double>(value)),
                            kExitInvalidInput);
            }
        }
    }
    for (int64_t k = 0; k < K; ++k) {
        for (int64_t j = 0; j < D; ++j) {
            const float value = C0[static_cast<size_t>(k * D + j)];
            if (!std::isfinite(value)) {
                return fail("VAL-006", "initial centroid at (row " + istr(k) + ", column " + istr(j) +
                                           ") is not finite: " + dstr(static_cast<double>(value)),
                            kExitInvalidInput);
            }
        }
    }

    // VAL-007: with M the largest magnitude in X and C0, require 4 D M^2 <= 2^126 so that no
    // binary32 distance can overflow during the run. Later centroids are means of points or
    // retained initial centroids, so they stay within [-M, M] up to rounding.
    const double m = range_bound(X, C0);
    const double bound = std::ldexp(1.0, 126);
    const double value = 4.0 * static_cast<double>(D) * m * m;
    if (!(value <= bound)) {
        return fail("VAL-007", "range bound exceeded: M=" + dstr(m) + ", 4*D*M^2=" + dstr(value) +
                                   " > 2^126=" + dstr(bound),
                    kExitInvalidInput);
    }
    if (m_range != nullptr) {
        *m_range = m;
    }

    // VAL-008: duplicate initial centroids are accepted with a warning. Compared bit-wise on the
    // float values, which is what the tie rule of DEC-005 sees.
    std::map<std::string, int64_t> seen;
    for (int64_t k = 0; k < K; ++k) {
        std::string key(reinterpret_cast<const char *>(&C0[static_cast<size_t>(k * D)]),
                        static_cast<size_t>(D) * sizeof(float));
        if (seen.count(key) != 0) {
            v.warnings.push_back("W-DUPC");
            break;
        }
        seen.emplace(std::move(key), k);
    }
    return v;
}

Validation validate_capacity(int64_t N, int64_t D, int64_t K, int64_t reduce_chunks,
                             uint64_t free_bytes)
{
    const MemoryModel m = device_memory_bytes(N, D, K, reduce_chunks);
    if (m.overflow) {
        // EDGE-30: index products near or above 2^63 are rejected by the checked arithmetic.
        return fail("VAL-009",
                    "required device memory overflows 64-bit arithmetic for N=" + istr(N) +
                        " D=" + istr(D) + " K=" + istr(K) + " G=" + istr(reduce_chunks),
                    kExitInvalidInput);
    }
    // A 5 % safety margin keeps allocation metadata and the context out of the way.
    const uint64_t budget = static_cast<uint64_t>(static_cast<double>(free_bytes) * 0.95);
    if (m.total > budget) {
        return fail("VAL-009",
                    "required " + ustr(m.total) + " B exceeds the usable " +
                        ustr(budget) + " B (95 % of " +
                        ustr(free_bytes) + " B free)",
                    kExitUnsupportedDevice);
    }
    return Validation{};
}

}  // namespace kmeans


In [ ]:
%%writefile io_kmd.h
// io_kmd.h — binary dataset and output files (requirements.md BLD-004, FR-002, FR-015).
//
// .kmd: magic "KMD1", uint32 version (= 1), uint64 rows, uint64 cols, then rows*cols binary32
//       little-endian in row-major order. Used for data and for centroids.
// .kma: magic "KMA1", uint32 version (= 1), uint64 N, then N int32 assignments.
//
// Every stream is opened with std::ios::binary. On Windows the default text mode translates byte
// 0x0A on write and 0x0D 0x0A on read, which silently corrupts float payloads containing those
// bytes, and the exact-size check below would then fail on a file that was written correctly.

#ifndef KMEANS_IO_KMD_H
#define KMEANS_IO_KMD_H

#include <cstdint>
#include <string>
#include <vector>

#include "common.h"

namespace kmeans {

struct Matrix {
    uint64_t rows = 0;
    uint64_t cols = 0;
    std::vector<float> data;  // row-major, rows*cols values
};

// BLD-004 requires little-endian on disk; the host must therefore be little-endian.
inline bool host_is_little_endian()
{
    const uint32_t one = 1;
    unsigned char bytes[sizeof(one)];
    std::memcpy(bytes, &one, sizeof(one));
    return bytes[0] == 1;
}

// Returns kExitOk, or kExitIoError for a missing, unreadable, truncated, over-long or
// structurally invalid file (bad magic or version). `err` receives a BLD-006 style message.
int read_kmd(const std::string &path, Matrix *out, std::string *err);
int read_kma(const std::string &path, std::vector<int32_t> *out, std::string *err);

// As read_kmd, plus the FR-002 check that the header dimensions equal the declared ones. A
// dimension mismatch is a validation failure (kExitInvalidInput), not an I/O failure.
int read_kmd_expect(const std::string &path, uint64_t rows, uint64_t cols, Matrix *out,
                    std::string *err);

int write_kmd(const std::string &path, uint64_t rows, uint64_t cols, const float *data,
              std::string *err);
int write_kma(const std::string &path, const std::vector<int32_t> &assignments, std::string *err);

}  // namespace kmeans

#endif  // KMEANS_IO_KMD_H


In [ ]:
%%writefile io_kmd.cpp
#include "io_kmd.h"

#include <fstream>

namespace kmeans {
namespace {

constexpr uint32_t kFormatVersion = 1;
constexpr size_t kKmdHeaderBytes = 4 + 4 + 8 + 8;  // magic, version, rows, cols
constexpr size_t kKmaHeaderBytes = 4 + 4 + 8;      // magic, version, N

int io_fail(std::string *err, const std::string &message)
{
    if (err != nullptr) {
        *err = message;
    }
    return kExitIoError;
}

// Opens the file, returns its exact size and leaves the stream positioned at the start.
bool open_sized(const std::string &path, std::ifstream *in, uint64_t *size)
{
    in->open(path, std::ios::binary | std::ios::ate);
    if (!in->is_open()) {
        return false;
    }
    const std::streampos end = in->tellg();
    if (end < 0) {
        return false;
    }
    *size = static_cast<uint64_t>(end);
    in->seekg(0, std::ios::beg);
    return true;
}

bool read_exact(std::ifstream *in, void *dst, size_t bytes)
{
    in->read(static_cast<char *>(dst), static_cast<std::streamsize>(bytes));
    return static_cast<size_t>(in->gcount()) == bytes && !in->bad();
}

}  // namespace

int read_kmd(const std::string &path, Matrix *out, std::string *err)
{
    if (!host_is_little_endian()) {
        return io_fail(err, "host is big-endian; BLD-004 fixes little-endian byte order on disk");
    }

    std::ifstream in;
    uint64_t file_size = 0;
    if (!open_sized(path, &in, &file_size)) {
        return io_fail(err, "cannot open '" + path + "' for reading");
    }
    if (file_size < kKmdHeaderBytes) {
        return io_fail(err, "'" + path + "' is " + std::to_string(file_size) +
                                " B, shorter than the " + std::to_string(kKmdHeaderBytes) +
                                " B .kmd header");
    }

    char magic[4] = {0, 0, 0, 0};
    uint32_t version = 0;
    uint64_t rows = 0;
    uint64_t cols = 0;
    if (!read_exact(&in, magic, 4) || !read_exact(&in, &version, 4) ||
        !read_exact(&in, &rows, 8) || !read_exact(&in, &cols, 8)) {
        return io_fail(err, "'" + path + "': header could not be read");
    }
    if (std::memcmp(magic, "KMD1", 4) != 0) {
        return io_fail(err, "'" + path + "': bad magic, expected KMD1");
    }
    if (version != kFormatVersion) {
        return io_fail(err, "'" + path + "': unsupported .kmd version " + std::to_string(version));
    }

    // FR-002: the file size must equal header + payload exactly, so a truncated file and a file
    // with trailing bytes are both rejected rather than silently accepted.
    uint64_t values = 0;
    uint64_t payload = 0;
    if (!checked_mul(rows, cols, &values) || !checked_mul(values, 4, &payload)) {
        return io_fail(err, "'" + path + "': header dimensions " + std::to_string(rows) + "x" +
                                std::to_string(cols) + " overflow the payload size");
    }
    const uint64_t expected = kKmdHeaderBytes + payload;
    if (file_size != expected) {
        return io_fail(err, "'" + path + "': size is " + std::to_string(file_size) +
                                " B but the header implies " + std::to_string(expected) + " B (" +
                                std::to_string(rows) + "x" + std::to_string(cols) + " binary32)");
    }

    out->rows = rows;
    out->cols = cols;
    out->data.assign(static_cast<size_t>(values), 0.0f);
    if (values != 0 && !read_exact(&in, out->data.data(), static_cast<size_t>(payload))) {
        return io_fail(err, "'" + path + "': payload could not be read");
    }
    return kExitOk;
}

int read_kma(const std::string &path, std::vector<int32_t> *out, std::string *err)
{
    if (!host_is_little_endian()) {
        return io_fail(err, "host is big-endian; BLD-004 fixes little-endian byte order on disk");
    }

    std::ifstream in;
    uint64_t file_size = 0;
    if (!open_sized(path, &in, &file_size)) {
        return io_fail(err, "cannot open '" + path + "' for reading");
    }
    if (file_size < kKmaHeaderBytes) {
        return io_fail(err, "'" + path + "' is shorter than the .kma header");
    }

    char magic[4] = {0, 0, 0, 0};
    uint32_t version = 0;
    uint64_t count = 0;
    if (!read_exact(&in, magic, 4) || !read_exact(&in, &version, 4) ||
        !read_exact(&in, &count, 8)) {
        return io_fail(err, "'" + path + "': header could not be read");
    }
    if (std::memcmp(magic, "KMA1", 4) != 0) {
        return io_fail(err, "'" + path + "': bad magic, expected KMA1");
    }
    if (version != kFormatVersion) {
        return io_fail(err, "'" + path + "': unsupported .kma version " + std::to_string(version));
    }

    uint64_t payload = 0;
    if (!checked_mul(count, 4, &payload)) {
        return io_fail(err, "'" + path + "': N=" + std::to_string(count) + " overflows the payload");
    }
    const uint64_t expected = kKmaHeaderBytes + payload;
    if (file_size != expected) {
        return io_fail(err, "'" + path + "': size is " + std::to_string(file_size) +
                                " B but the header implies " + std::to_string(expected) + " B");
    }

    out->assign(static_cast<size_t>(count), 0);
    if (count != 0 && !read_exact(&in, out->data(), static_cast<size_t>(payload))) {
        return io_fail(err, "'" + path + "': payload could not be read");
    }
    return kExitOk;
}

int read_kmd_expect(const std::string &path, uint64_t rows, uint64_t cols, Matrix *out,
                    std::string *err)
{
    const int code = read_kmd(path, out, err);
    if (code != kExitOk) {
        return code;
    }
    if (out->rows != rows || out->cols != cols) {
        if (err != nullptr) {
            *err = "'" + path + "' holds " + std::to_string(out->rows) + "x" +
                   std::to_string(out->cols) + " values but " + std::to_string(rows) + "x" +
                   std::to_string(cols) + " was declared";
        }
        return kExitInvalidInput;  // shape mismatch is a validation failure (FR-002)
    }
    return kExitOk;
}

int write_kmd(const std::string &path, uint64_t rows, uint64_t cols, const float *data,
              std::string *err)
{
    std::ofstream out(path, std::ios::binary | std::ios::trunc);
    if (!out.is_open()) {
        return io_fail(err, "cannot open '" + path + "' for writing");
    }
    const uint32_t version = kFormatVersion;
    out.write("KMD1", 4);
    out.write(reinterpret_cast<const char *>(&version), 4);
    out.write(reinterpret_cast<const char *>(&rows), 8);
    out.write(reinterpret_cast<const char *>(&cols), 8);

    uint64_t values = 0;
    if (!checked_mul(rows, cols, &values)) {
        return io_fail(err, "'" + path + "': dimensions overflow the payload size");
    }
    if (values != 0) {
        out.write(reinterpret_cast<const char *>(data),
                  static_cast<std::streamsize>(values * sizeof(float)));
    }
    out.flush();
    if (!out.good()) {
        return io_fail(err, "'" + path + "': write failed");
    }
    return kExitOk;
}

int write_kma(const std::string &path, const std::vector<int32_t> &assignments, std::string *err)
{
    std::ofstream out(path, std::ios::binary | std::ios::trunc);
    if (!out.is_open()) {
        return io_fail(err, "cannot open '" + path + "' for writing");
    }
    const uint32_t version = kFormatVersion;
    const uint64_t count = assignments.size();
    out.write("KMA1", 4);
    out.write(reinterpret_cast<const char *>(&version), 4);
    out.write(reinterpret_cast<const char *>(&count), 8);
    if (count != 0) {
        out.write(reinterpret_cast<const char *>(assignments.data()),
                  static_cast<std::streamsize>(count * sizeof(int32_t)));
    }
    out.flush();
    if (!out.good()) {
        return io_fail(err, "'" + path + "': write failed");
    }
    return kExitOk;
}

}  // namespace kmeans


In [ ]:
%%writefile cpu_reference.h
// cpu_reference.h — single-threaded CPU oracle (requirements.md NUM-003, ARC-03).
//
// Implements §2 exactly: the binary32 distance path of §2.2 with ascending-j accumulation, a
// binary64 path used only for near-tie diagnosis (NUM-005), binary64 sums in ascending i (§2.4),
// the lowest-index tie rule (DEC-005), empty-cluster retention (DEC-003), the convergence rule
// (DEC-002), iteration counting (DEC-006) and finalization (§2.8).
//
// The step functions are exposed individually, not only through run(), because §10.3 lockstep
// verification feeds the device's own state into each step and compares the result.
//
// Must be compiled with -ffp-contract=off and without fast-math: contracting a multiply-add in
// distance2_fp32 would change the oracle's rounding and make near-tie diagnosis meaningless.

#ifndef KMEANS_CPU_REFERENCE_H
#define KMEANS_CPU_REFERENCE_H

#include <cstdint>
#include <vector>

#include "common.h"

namespace kmeans {
namespace cpu {

// §2.2 in binary32: terms and accumulation in binary32, ascending j, never reordered.
float distance2_fp32(const float *x, const float *c, int64_t D);

// The same quantity in binary64, for NUM-005 margins and for the objective (§2.9). Not used for
// assignment: that must see exactly what the device sees.
double distance2_fp64(const float *x, const float *c, int64_t D);

// §2.3 for one point: nearest centroid by binary32 distance, lowest index among exactly equal
// computed values (strict-less-than replacement in ascending k).
int32_t nearest_centroid(const float *x, int64_t D, const float *C, int64_t K);

// §2.3 for all points, writing A. Returns the number of points whose assignment differs from
// A_prev (FR-007). With A_prev == nullptr this is the initial pass and the return value is N,
// which the driver ignores because changes are only counted inside the loop.
int64_t assign(const float *X, int64_t N, int64_t D, const float *C, int64_t K, int32_t *A,
               const int32_t *A_prev);

// NUM-005 diagnosis for one point, in binary64.
struct TieInfo {
    double d1 = 0.0;      // smallest binary64 distance
    double d2 = 0.0;      // second smallest
    double margin = 0.0;  // (d2 - d1) / max(d2, FLT_MIN)
    int32_t k1 = 0;       // index attaining d1 (lowest index on ties)
    int32_t k2 = 0;       // index attaining d2
    bool near_tie = false;
};
TieInfo tie_info(const float *x, int64_t D, const float *C, int64_t K);

// §2.4: counts exactly, sums in binary64 in ascending i. S has K*D entries, n has K.
void accumulate(const float *X, int64_t N, int64_t D, const int32_t *A, int64_t K, double *S,
                uint32_t *n);

// §2.5: C_next[k] = fl32(S[k]/n[k]) for n[k] > 0, else C_cur[k] retained bit-exactly. No division
// is performed when n[k] == 0. C_next must differ from C_cur (double buffering, FR-010).
void update(const double *S, const uint32_t *n, int64_t K, int64_t D, const float *C_cur,
            float *C_next);

// §2.6: max over clusters with n[k] > 0 of the binary64 Euclidean displacement. Empty clusters
// contribute delta_k = 0 by construction and are excluded from the maximum.
double movement(const float *C_cur, const float *C_next, const uint32_t *n, int64_t K, int64_t D);

// §2.9: J in binary64, computed after the run and outside any timed region (FR-016).
double objective(const float *X, int64_t N, int64_t D, const int32_t *A, const float *C);

// §2.8 steps 1 and 2: one accumulation pass over A_final, then the P2 check against NUM-006.
// C_final is never modified. `m_range` is the M of VAL-007, which fixes the NUM-006 scale.
struct Finalization {
    std::vector<uint32_t> counts;       // n_final
    std::vector<float> means;           // fl32(S/n) for non-empty k, C_final retained otherwise
    bool centroid_consistent = false;   // P2
    double max_centroid_difference = 0.0;
};
Finalization finalize(const float *X, int64_t N, int64_t D, const int32_t *A_final,
                      const float *C_final, int64_t K, double m_range);

// §2.7 driver. `fixed_iters` (FR-020) disables both convergence criteria so exactly max_iter
// updates run and the status is ITERATION_LIMIT.
Result run(const float *X, int64_t N, int64_t D, const float *C0, int64_t K, int64_t max_iter,
           double tol, double m_range, bool fixed_iters);

}  // namespace cpu
}  // namespace kmeans

#endif  // KMEANS_CPU_REFERENCE_H


In [ ]:
%%writefile cpu_reference.cpp
#include "cpu_reference.h"

#include <cfloat>
#include <cmath>

namespace kmeans {
namespace cpu {

float distance2_fp32(const float *x, const float *c, int64_t D)
{
    // NUM-002: each term is formed in binary32 and accumulated in binary32 in ascending j,
    // starting from 0. The order is never changed, and -ffp-contract=off keeps the compiler from
    // fusing the multiply and add, so this is the exact value the device must reproduce.
    float acc = 0.0f;
    for (int64_t j = 0; j < D; ++j) {
        const float diff = x[j] - c[j];
        acc += diff * diff;
    }
    return acc;
}

double distance2_fp64(const float *x, const float *c, int64_t D)
{
    double acc = 0.0;
    for (int64_t j = 0; j < D; ++j) {
        const double diff = static_cast<double>(x[j]) - static_cast<double>(c[j]);
        acc += diff * diff;
    }
    return acc;
}

int32_t nearest_centroid(const float *x, int64_t D, const float *C, int64_t K)
{
    // DEC-005: start at k = 0 and replace only on strictly smaller distance, so exactly equal
    // computed binary32 distances keep the lowest index.
    int32_t best_k = 0;
    float best_d = distance2_fp32(x, C, D);
    for (int64_t k = 1; k < K; ++k) {
        const float d = distance2_fp32(x, C + k * D, D);
        if (d < best_d) {
            best_d = d;
            best_k = static_cast<int32_t>(k);
        }
    }
    return best_k;
}

int64_t assign(const float *X, int64_t N, int64_t D, const float *C, int64_t K, int32_t *A,
               const int32_t *A_prev)
{
    int64_t changed = 0;
    for (int64_t i = 0; i < N; ++i) {
        const int32_t k = nearest_centroid(X + i * D, D, C, K);
        if (A_prev != nullptr && A_prev[i] != k) {
            ++changed;
        }
        A[i] = k;
    }
    return A_prev == nullptr ? N : changed;
}

TieInfo tie_info(const float *x, int64_t D, const float *C, int64_t K)
{
    TieInfo info;
    info.d1 = HUGE_VAL;
    info.d2 = HUGE_VAL;
    info.k1 = 0;
    info.k2 = K > 1 ? 1 : 0;

    for (int64_t k = 0; k < K; ++k) {
        const double d = distance2_fp64(x, C + k * D, D);
        if (d < info.d1) {
            info.d2 = info.d1;
            info.k2 = info.k1;
            info.d1 = d;
            info.k1 = static_cast<int32_t>(k);
        } else if (d < info.d2) {
            info.d2 = d;
            info.k2 = static_cast<int32_t>(k);
        }
    }

    if (K < 2) {
        // With one centroid there is no competitor, so no point can be a near-tie.
        info.d2 = info.d1;
        info.margin = HUGE_VAL;
        info.near_tie = false;
        return info;
    }
    const double denom = info.d2 > static_cast<double>(FLT_MIN) ? info.d2
                                                                : static_cast<double>(FLT_MIN);
    info.margin = (info.d2 - info.d1) / denom;
    info.near_tie = info.margin <= near_tie_threshold(D);
    return info;
}

void accumulate(const float *X, int64_t N, int64_t D, const int32_t *A, int64_t K, double *S,
                uint32_t *n)
{
    for (int64_t e = 0; e < K * D; ++e) {
        S[e] = 0.0;
    }
    for (int64_t k = 0; k < K; ++k) {
        n[k] = 0;
    }
    // §2.4: ascending i, binary64 sums, exact integer counts. The reference fixes this order so
    // that Approach B (which also has a fixed order) can be compared against it meaningfully.
    for (int64_t i = 0; i < N; ++i) {
        const int64_t k = A[i];
        n[k] += 1;
        const float *row = X + i * D;
        double *sums = S + k * D;
        for (int64_t j = 0; j < D; ++j) {
            sums[j] += static_cast<double>(row[j]);
        }
    }
}

void update(const double *S, const uint32_t *n, int64_t K, int64_t D, const float *C_cur,
            float *C_next)
{
    for (int64_t k = 0; k < K; ++k) {
        const float *cur = C_cur + k * D;
        float *next = C_next + k * D;
        if (n[k] == 0) {
            // DEC-003: retain the previous centroid bit-exactly; no division is performed.
            for (int64_t j = 0; j < D; ++j) {
                next[j] = cur[j];
            }
            continue;
        }
        const double count = static_cast<double>(n[k]);
        const double *sums = S + k * D;
        for (int64_t j = 0; j < D; ++j) {
            // §2.5: divide in binary64 and round the quotient once to binary32.
            next[j] = static_cast<float>(sums[j] / count);
        }
    }
}

double movement(const float *C_cur, const float *C_next, const uint32_t *n, int64_t K, int64_t D)
{
    double max_delta = 0.0;
    for (int64_t k = 0; k < K; ++k) {
        if (n[k] == 0) {
            continue;  // §2.6: empty clusters have delta_k = 0 and are excluded from the maximum
        }
        const float *cur = C_cur + k * D;
        const float *next = C_next + k * D;
        double sum = 0.0;
        for (int64_t j = 0; j < D; ++j) {
            const double diff = static_cast<double>(next[j]) - static_cast<double>(cur[j]);
            sum += diff * diff;
        }
        const double delta = std::sqrt(sum);
        if (delta > max_delta) {
            max_delta = delta;
        }
    }
    return max_delta;
}

double objective(const float *X, int64_t N, int64_t D, const int32_t *A, const float *C)
{
    double j_total = 0.0;
    for (int64_t i = 0; i < N; ++i) {
        j_total += distance2_fp64(X + i * D, C + static_cast<int64_t>(A[i]) * D, D);
    }
    return j_total;
}

Finalization finalize(const float *X, int64_t N, int64_t D, const int32_t *A_final,
                      const float *C_final, int64_t K, double m_range)
{
    Finalization out;
    out.counts.assign(static_cast<size_t>(K), 0);
    out.means.assign(static_cast<size_t>(K * D), 0.0f);

    std::vector<double> S(static_cast<size_t>(K * D), 0.0);
    accumulate(X, N, D, A_final, K, S.data(), out.counts.data());
    update(S.data(), out.counts.data(), K, D, C_final, out.means.data());

    // §2.8 step 2: P2 holds when every non-empty cluster's reported centroid agrees with the mean
    // of its assigned points within NUM-006. Empty clusters are not evaluated: their centroid is
    // retained by DEC-003 and is not a mean of anything.
    const double s_max = s_max_from_range(m_range);
    bool consistent = true;
    double worst = 0.0;
    for (int64_t k = 0; k < K; ++k) {
        if (out.counts[static_cast<size_t>(k)] == 0) {
            continue;
        }
        for (int64_t j = 0; j < D; ++j) {
            const size_t e = static_cast<size_t>(k * D + j);
            const float reported = C_final[e];
            const float mean = out.means[e];
            const double diff =
                std::fabs(static_cast<double>(reported) - static_cast<double>(mean));
            if (diff > worst) {
                worst = diff;
            }
            if (!centroid_agrees(reported, mean, s_max)) {
                consistent = false;
            }
        }
    }
    out.centroid_consistent = consistent;
    out.max_centroid_difference = worst;
    return out;
}

Result run(const float *X, int64_t N, int64_t D, const float *C0, int64_t K, int64_t max_iter,
           double tol, double m_range, bool fixed_iters)
{
    Result result;
    std::vector<float> c_cur(C0, C0 + K * D);
    std::vector<float> c_next(static_cast<size_t>(K * D), 0.0f);
    std::vector<int32_t> a_cur(static_cast<size_t>(N), 0);
    std::vector<int32_t> a_prev(static_cast<size_t>(N), 0);
    std::vector<double> sums(static_cast<size_t>(K * D), 0.0);
    std::vector<uint32_t> counts(static_cast<size_t>(K), 0);

    // §2.7: one assignment pass before the loop, so A is always the assignment of the current C.
    assign(X, N, D, c_cur.data(), K, a_cur.data(), nullptr);

    int64_t t = 0;
    Status status = Status::ITERATION_LIMIT;
    double last_delta = 0.0;
    bool has_last_delta = false;

    while (t < max_iter) {
        a_prev = a_cur;

        accumulate(X, N, D, a_cur.data(), K, sums.data(), counts.data());
        update(sums.data(), counts.data(), K, D, c_cur.data(), c_next.data());
        const double delta = movement(c_cur.data(), c_next.data(), counts.data(), K, D);
        c_cur.swap(c_next);  // FR-010: double buffering, pointers swapped after the update

        ++t;  // DEC-006: the iteration count is the number of completed centroid updates
        last_delta = delta;
        has_last_delta = true;

        const int64_t changed = assign(X, N, D, c_cur.data(), K, a_cur.data(), a_prev.data());

        if (fixed_iters) {
            continue;  // FR-020: both criteria disabled, so exactly max_iter updates run
        }
        // DEC-002: assignment stability is checked first and takes reporting precedence.
        if (changed == 0) {
            status = Status::CONVERGED_ASSIGNMENTS;
            break;
        }
        if (delta <= tol) {
            status = Status::CONVERGED_MOVEMENT;  // inclusive comparison (ENV-10)
            break;
        }
    }

    const Finalization fin = finalize(X, N, D, a_cur.data(), c_cur.data(), K, m_range);

    result.assignments = a_cur;
    result.centroids = c_cur;  // §2.8: finalization never replaces C_final
    result.counts = fin.counts;
    result.iterations = t;
    result.status = status;
    result.last_delta = last_delta;
    result.has_last_delta = has_last_delta;
    result.centroid_consistent = fin.centroid_consistent;
    result.assignment_consistent = true;  // P1 holds by construction
    result.objective = objective(X, N, D, a_cur.data(), c_cur.data());
    return result;
}

}  // namespace cpu
}  // namespace kmeans


In [ ]:
%%writefile gen.h
// gen.h — deterministic Gaussian-blob dataset generator (requirements.md FR-003, ARC-02).
//
// Uses the raw output of std::mt19937_64 plus a self-implemented Box-Muller transform. Standard
// library distributions (std::normal_distribution, std::uniform_int_distribution) are deliberately
// not used: their algorithms are unspecified, so the same seed can yield different numbers on a
// different standard library, and FR-003 requires identical parameters to reproduce bit-identical
// files.

#ifndef KMEANS_GEN_H
#define KMEANS_GEN_H

#include <cstdint>
#include <random>
#include <string>
#include <vector>

namespace kmeans {

struct GenParams {
    int64_t N = 0;
    int64_t D = 0;
    int64_t K = 0;        // K_true = K (PERF-003)
    double sep = 10.0;    // scale of the cluster-centre spread
    double spread = 1.0;  // standard deviation of the points around their centre
    uint64_t seed = 0;
};

// Raw-output wrapper. Every draw below consumes a documented, fixed number of 64-bit words, so the
// generation order fixes the stream and the output is reproducible.
class Rng {
public:
    explicit Rng(uint64_t seed) : engine_(seed) {}

    // Uniform in the open interval (0, 1): the top 53 bits plus a half-step, so the result is
    // never exactly 0 (which would make std::log in Box-Muller return -inf) and never exactly 1.
    double uniform_open();

    // Standard normal via Box-Muller. Both values of each pair are used, so two normals cost one
    // pair of uniforms; the second is cached.
    double normal();

    // Uniform integer in [0, bound) by rejection, which is unbiased for every bound.
    uint64_t bounded(uint64_t bound);

private:
    std::mt19937_64 engine_;
    bool has_cached_normal_ = false;
    double cached_normal_ = 0.0;
};

struct Dataset {
    std::vector<float> X;    // N x D row-major
    std::vector<float> C0;   // K x D row-major
    std::string dataset_id;
    double min_centre_distance = 0.0;  // smallest pairwise centre distance, measured not assumed
    double spread = 0.0;               // echoed so the caller can report separation in sigma units
};

// Identifier derived from every generator parameter (FR-003), so a dataset is traceable from a
// benchmark CSV row back to the exact inputs that produced it.
std::string dataset_id(const GenParams &p);

// Generation order, fixed by FR-003 and by this function:
//   1. K centre vectors, each coordinate sep * normal()
//   2. N points: point i belongs to centre i % K and gets spread * normal() per coordinate
//   3. a seeded Fisher-Yates permutation of [0, N)
//   4. initial centroids: the points at permutation indices floor(q*N/K), q in [0, K)
Dataset generate(const GenParams &p);

}  // namespace kmeans

#endif  // KMEANS_GEN_H


In [ ]:
%%writefile gen.cpp
#include "gen.h"

#include <cmath>
#include <cstdio>

namespace kmeans {
namespace {

constexpr double kTwoPi = 6.283185307179586476925286766559;

// FNV-1a over the parameter string, so a dataset ID is short but still parameter-specific.
uint64_t fnv1a(const std::string &text)
{
    uint64_t hash = 1469598103934665603ull;
    for (unsigned char c : text) {
        hash ^= static_cast<uint64_t>(c);
        hash *= 1099511628211ull;
    }
    return hash;
}

}  // namespace

double Rng::uniform_open()
{
    // 53 significant bits is the most a binary64 can hold; adding half a step moves the value off
    // both endpoints without biasing the distribution's mean.
    const uint64_t raw = engine_();
    return (static_cast<double>(raw >> 11) + 0.5) * (1.0 / 9007199254740992.0);  // 2^-53
}

double Rng::normal()
{
    if (has_cached_normal_) {
        has_cached_normal_ = false;
        return cached_normal_;
    }
    const double u1 = uniform_open();
    const double u2 = uniform_open();
    const double radius = std::sqrt(-2.0 * std::log(u1));
    const double angle = kTwoPi * u2;
    cached_normal_ = radius * std::sin(angle);
    has_cached_normal_ = true;
    return radius * std::cos(angle);
}

uint64_t Rng::bounded(uint64_t bound)
{
    if (bound <= 1) {
        return 0;
    }
    // Rejection sampling: discard the incomplete final block of the 2^64 range so every value in
    // [0, bound) is equally likely. std::uniform_int_distribution would do this too, but its
    // algorithm is implementation-defined and FR-003 needs a fixed one.
    const uint64_t limit = UINT64_MAX - (UINT64_MAX % bound) - 1;
    uint64_t raw = engine_();
    while (raw > limit) {
        raw = engine_();
    }
    return raw % bound;
}

std::string dataset_id(const GenParams &p)
{
    char buf[256];
    std::snprintf(buf, sizeof(buf), "n%lld_d%lld_k%lld_sep%.6g_spread%.6g_seed%llu",
                  static_cast<long long>(p.N), static_cast<long long>(p.D),
                  static_cast<long long>(p.K), p.sep, p.spread,
                  static_cast<unsigned long long>(p.seed));
    const std::string base(buf);
    char suffix[32];
    std::snprintf(suffix, sizeof(suffix), "-%016llx",
                  static_cast<unsigned long long>(fnv1a(base)));
    return base + suffix;
}

Dataset generate(const GenParams &p)
{
    Dataset out;
    out.dataset_id = dataset_id(p);
    out.spread = p.spread;
    const int64_t N = p.N;
    const int64_t D = p.D;
    const int64_t K = p.K;

    Rng rng(p.seed);

    // 1. Cluster centres. Drawing them from the same normal stream keeps the generator simple and
    // deterministic; the realised separation is measured below rather than assumed, because
    // §10.3 calls a dataset well-separated only at separation >= 10 * spread.
    std::vector<double> centres(static_cast<size_t>(K * D), 0.0);
    for (int64_t k = 0; k < K; ++k) {
        for (int64_t j = 0; j < D; ++j) {
            centres[static_cast<size_t>(k * D + j)] = p.sep * rng.normal();
        }
    }

    // 2. Points: point i belongs to centre i % K, which spreads the clusters evenly without
    // consuming randomness for the membership choice.
    out.X.assign(static_cast<size_t>(N * D), 0.0f);
    for (int64_t i = 0; i < N; ++i) {
        const int64_t k = i % K;
        for (int64_t j = 0; j < D; ++j) {
            const double value = centres[static_cast<size_t>(k * D + j)] + p.spread * rng.normal();
            out.X[static_cast<size_t>(i * D + j)] = static_cast<float>(value);
        }
    }

    // 3. Fisher-Yates permutation of the point indices, descending as in the classical algorithm.
    std::vector<int64_t> perm(static_cast<size_t>(N));
    for (int64_t i = 0; i < N; ++i) {
        perm[static_cast<size_t>(i)] = i;
    }
    for (int64_t i = N - 1; i > 0; --i) {
        const uint64_t j = rng.bounded(static_cast<uint64_t>(i) + 1);
        const int64_t tmp = perm[static_cast<size_t>(i)];
        perm[static_cast<size_t>(i)] = perm[static_cast<size_t>(j)];
        perm[static_cast<size_t>(j)] = tmp;
    }

    // 4. FR-003: the initial centroids are the points at permutation indices floor(q*N/K). With
    // K > N the same point can be chosen more than once, which VAL-008 then flags as W-DUPC.
    out.C0.assign(static_cast<size_t>(K * D), 0.0f);
    for (int64_t q = 0; q < K; ++q) {
        const int64_t slot = static_cast<int64_t>((static_cast<uint64_t>(q) *
                                                   static_cast<uint64_t>(N)) /
                                                  static_cast<uint64_t>(K));
        const int64_t point = perm[static_cast<size_t>(slot % N)];
        for (int64_t j = 0; j < D; ++j) {
            out.C0[static_cast<size_t>(q * D + j)] = out.X[static_cast<size_t>(point * D + j)];
        }
    }

    // Measured minimum pairwise centre distance, in the generated binary32 values' own units.
    double min_sq = -1.0;
    for (int64_t a = 0; a < K; ++a) {
        for (int64_t b = a + 1; b < K; ++b) {
            double sum = 0.0;
            for (int64_t j = 0; j < D; ++j) {
                const double diff = centres[static_cast<size_t>(a * D + j)] -
                                    centres[static_cast<size_t>(b * D + j)];
                sum += diff * diff;
            }
            if (min_sq < 0.0 || sum < min_sq) {
                min_sq = sum;
            }
        }
    }
    out.min_centre_distance = min_sq < 0.0 ? 0.0 : std::sqrt(min_sq);
    return out;
}

}  // namespace kmeans


In [ ]:
%%writefile gen_main.cpp
// gen_main.cpp — the kmeans-gen executable (requirements.md BLD-003a, FR-003).
//
//   kmeans-gen --n N --d D --k K --sep S --spread SIGMA --seed SEED --out-prefix PATH
//
// Writes PATH_X.kmd (N x D) and PATH_C0.kmd (K x D) in the BLD-004 format and prints a one-line
// JSON summary including the dataset ID and the measured minimum centre separation, so a benchmark
// row can be traced back to the exact dataset (PERF-008) and the "well-separated" claim of §10.3
// is a measurement rather than an assumption.

#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>

#include "common.h"
#include "gen.h"
#include "io_kmd.h"
#include "validate.h"

namespace {

void usage()
{
    std::fprintf(stderr,
                 "usage: kmeans-gen --n N --d D --k K --sep S --spread SIGMA --seed SEED "
                 "--out-prefix PATH\n");
}

bool parse_int64(const char *text, int64_t *out)
{
    char *end = nullptr;
    const long long value = std::strtoll(text, &end, 10);
    if (end == text || *end != '\0') {
        return false;
    }
    *out = static_cast<int64_t>(value);
    return true;
}

bool parse_uint64(const char *text, uint64_t *out)
{
    char *end = nullptr;
    const unsigned long long value = std::strtoull(text, &end, 10);
    if (end == text || *end != '\0') {
        return false;
    }
    *out = static_cast<uint64_t>(value);
    return true;
}

bool parse_double(const char *text, double *out)
{
    char *end = nullptr;
    const double value = std::strtod(text, &end);
    if (end == text || *end != '\0') {
        return false;
    }
    *out = value;
    return true;
}

}  // namespace

int main(int argc, char **argv)
{
    using namespace kmeans;

    GenParams params;
    std::string out_prefix;
    bool have_n = false, have_d = false, have_k = false;

    for (int i = 1; i < argc; ++i) {
        const char *flag = argv[i];
        const bool needs_value = std::strncmp(flag, "--", 2) == 0;
        if (!needs_value) {
            report_error("VAL-010", "unexpected argument '" + std::string(flag) + "'");
            usage();
            return kExitInvalidInput;
        }
        if (i + 1 >= argc) {
            report_error("VAL-010", "option '" + std::string(flag) + "' needs a value");
            usage();
            return kExitInvalidInput;
        }
        const char *value = argv[++i];
        bool parsed = true;
        if (std::strcmp(flag, "--n") == 0) {
            parsed = parse_int64(value, &params.N);
            have_n = true;
        } else if (std::strcmp(flag, "--d") == 0) {
            parsed = parse_int64(value, &params.D);
            have_d = true;
        } else if (std::strcmp(flag, "--k") == 0) {
            parsed = parse_int64(value, &params.K);
            have_k = true;
        } else if (std::strcmp(flag, "--sep") == 0) {
            parsed = parse_double(value, &params.sep);
        } else if (std::strcmp(flag, "--spread") == 0) {
            parsed = parse_double(value, &params.spread);
        } else if (std::strcmp(flag, "--seed") == 0) {
            parsed = parse_uint64(value, &params.seed);
        } else if (std::strcmp(flag, "--out-prefix") == 0) {
            out_prefix = value;
        } else {
            report_error("VAL-010", "unknown option '" + std::string(flag) + "'");
            usage();
            return kExitInvalidInput;
        }
        if (!parsed) {
            report_error("VAL-010", "value '" + std::string(value) + "' for option '" +
                                        std::string(flag) + "' could not be parsed");
            return kExitInvalidInput;
        }
    }

    if (!have_n || !have_d || !have_k || out_prefix.empty()) {
        report_error("VAL-010", "--n, --d, --k and --out-prefix are required");
        usage();
        return kExitInvalidInput;
    }

    // The generator's sizes obey the same VAL-001/VAL-002 limits as a clustering run, so an
    // impossible dataset is refused before any memory is touched.
    const Validation params_ok = validate_parameters(params.N, params.D, params.K, 0, 0.0);
    if (!params_ok.ok) {
        params_ok.report();
        return params_ok.exit_code;
    }
    if (!(params.spread >= 0.0) || !(params.sep >= 0.0)) {
        report_error("VAL-010", "--sep and --spread must be non-negative and not NaN");
        return kExitInvalidInput;
    }

    const Dataset data = generate(params);

    std::string err;
    const std::string x_path = out_prefix + "_X.kmd";
    const std::string c_path = out_prefix + "_C0.kmd";
    int code = write_kmd(x_path, static_cast<uint64_t>(params.N), static_cast<uint64_t>(params.D),
                         data.X.data(), &err);
    if (code != kExitOk) {
        report_error("FR-002", err);
        return code;
    }
    code = write_kmd(c_path, static_cast<uint64_t>(params.K), static_cast<uint64_t>(params.D),
                     data.C0.data(), &err);
    if (code != kExitOk) {
        report_error("FR-002", err);
        return code;
    }

    const double sigma_ratio =
        params.spread > 0.0 ? data.min_centre_distance / params.spread : 0.0;
    std::printf("{\"dataset_id\": \"%s\", \"N\": %lld, \"D\": %lld, \"K\": %lld, \"sep\": %.17g, "
                "\"spread\": %.17g, \"seed\": %llu, \"min_centre_distance\": %.17g, "
                "\"min_separation_sigma\": %.17g, \"well_separated\": %s, \"data\": \"%s\", "
                "\"centroids\": \"%s\"}\n",
                data.dataset_id.c_str(), static_cast<long long>(params.N),
                static_cast<long long>(params.D), static_cast<long long>(params.K), params.sep,
                params.spread, static_cast<unsigned long long>(params.seed),
                data.min_centre_distance, sigma_ratio, sigma_ratio >= 10.0 ? "true" : "false",
                x_path.c_str(), c_path.c_str());
    return kExitOk;
}


In [ ]:
%%writefile hand_data.h
// hand_data.h — the hand-verifiable datasets H1-H6 and H2-a..H2-e of requirements.md §16.0,
// together with their expected outcomes.
//
// These are the oracle for the oracle: every value here is derived by hand in the specification,
// so a CPU reference that reproduces them is trustworthy enough to judge the device against
// (TEST-I01..I06 reuse exactly these cases on the GPU). All values except 5/3 are exactly
// representable in binary32 and are compared exactly; 5/3 is compared with NUM-006.

#ifndef KMEANS_TESTS_HAND_DATA_H
#define KMEANS_TESTS_HAND_DATA_H

#include <cstdint>
#include <string>
#include <vector>

#include "common.h"

namespace kmeans {
namespace tests {

struct HandCase {
    std::string id;
    int64_t N = 0;
    int64_t D = 0;
    int64_t K = 0;
    std::vector<float> X;
    std::vector<float> C0;
    int64_t max_iter = 10;
    double tol = 0.0;

    // Expected outcome (§16.0).
    std::vector<int32_t> expected_A;
    std::vector<float> expected_C;
    std::vector<uint32_t> expected_n;
    int64_t expected_iterations = 0;
    Status expected_status = Status::ERROR;
    bool expected_centroid_consistent = false;
    bool expect_warning_kgtn = false;  // VAL-003
    bool expect_warning_dupc = false;  // VAL-008
    bool centroids_exact = true;       // false where 5/3 appears, compared via NUM-006
    std::string note;
};

// The 1-D H2 point set, reused by H2 and its five variants.
inline std::vector<float> h2_points()
{
    return {0.0f, 2.0f, 3.0f, 10.0f};
}

inline std::vector<float> h2_initial_centroids()
{
    return {0.0f, 3.0f};
}

inline std::vector<HandCase> hand_cases()
{
    std::vector<HandCase> cases;

    // H1: two well-separated pairs. A(0) = {0,0,1,1}; C(1) = {0.5, 9.5}; A(1) = A(0).
    {
        HandCase c;
        c.id = "H1";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = {0.0f, 1.0f, 9.0f, 10.0f};
        c.C0 = {0.0f, 10.0f};
        c.expected_A = {0, 0, 1, 1};
        c.expected_C = {0.5f, 9.5f};
        c.expected_n = {2, 2};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        cases.push_back(c);
    }

    // H2: four updates of drift. A(0)={0,1,1,1}; C(1)={0,5} (delta 2); A(1)={0,0,1,1};
    // C(2)={1,6.5} (delta 1.5); A(2)={0,0,0,1}; C(3)={5/3,10} (delta 3.5); A(3)=A(2).
    {
        HandCase c;
        c.id = "H2";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {5.0f / 3.0f, 10.0f};
        c.expected_n = {3, 1};
        c.expected_iterations = 3;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.centroids_exact = false;  // 5/3 is not representable in binary32 (NUM-006)
        cases.push_back(c);
    }

    // H2-a: tolerance 2 equals the first delta exactly, so the inclusive comparison of DEC-002
    // stops after one update. P2 is false: cluster 0 now holds {0,2}, whose mean is 1, not 0.
    {
        HandCase c;
        c.id = "H2-a";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.tol = 2.0;
        c.expected_A = {0, 0, 1, 1};
        c.expected_C = {0.0f, 5.0f};
        c.expected_n = {2, 2};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_MOVEMENT;
        c.expected_centroid_consistent = false;
        c.note = "delta == tol exactly: inclusive comparison (ENV-10)";
        cases.push_back(c);
    }

    // H2-b: tolerance 1.5 equals the second delta exactly.
    {
        HandCase c;
        c.id = "H2-b";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.tol = 1.5;
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {1.0f, 6.5f};
        c.expected_n = {3, 1};
        c.expected_iterations = 2;
        c.expected_status = Status::CONVERGED_MOVEMENT;
        c.expected_centroid_consistent = false;
        cases.push_back(c);
    }

    // H2-c: just below that delta, so the run continues to assignment stability as in H2.
    {
        HandCase c;
        c.id = "H2-c";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.tol = 1.4999;
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {5.0f / 3.0f, 10.0f};
        c.expected_n = {3, 1};
        c.expected_iterations = 3;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.centroids_exact = false;
        cases.push_back(c);
    }

    // H2-d: the limit cuts the run short; termination is not reported as convergence.
    {
        HandCase c;
        c.id = "H2-d";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.max_iter = 2;
        c.expected_A = {0, 0, 0, 1};
        c.expected_C = {1.0f, 6.5f};
        c.expected_n = {3, 1};
        c.expected_iterations = 2;
        c.expected_status = Status::ITERATION_LIMIT;
        c.expected_centroid_consistent = false;
        cases.push_back(c);
    }

    // H2-e: max_iter = 0 runs the initial assignment pass only (§2.7 boundary behaviour).
    {
        HandCase c;
        c.id = "H2-e";
        c.N = 4; c.D = 1; c.K = 2;
        c.X = h2_points();
        c.C0 = h2_initial_centroids();
        c.max_iter = 0;
        c.expected_A = {0, 1, 1, 1};
        c.expected_C = {0.0f, 3.0f};
        c.expected_n = {1, 3};
        c.expected_iterations = 0;
        c.expected_status = Status::ITERATION_LIMIT;
        c.expected_centroid_consistent = false;  // cluster 1 holds {2,3,10}, mean 5, not 3
        cases.push_back(c);
    }

    // H3: one point exactly between two centroids. The tie goes to the lower index (DEC-005) and
    // cluster 1 is empty from the start, so it retains its initial centroid (DEC-003).
    {
        HandCase c;
        c.id = "H3";
        c.N = 1; c.D = 1; c.K = 2;
        c.X = {1.0f};
        c.C0 = {0.0f, 2.0f};
        c.expected_A = {0};
        c.expected_C = {1.0f, 2.0f};
        c.expected_n = {1, 0};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;  // the only non-empty cluster holds its own mean
        // K = 2 > N = 1, so VAL-003 applies here as well. The §16.0 row for H3 does not mention
        // the warning, but the rule is normative and unconditional.
        c.expect_warning_kgtn = true;
        c.note = "exact tie -> lowest index; empty cluster retained and excluded from delta";
        cases.push_back(c);
    }

    // H4: duplicate initial centroids. Every point ties and goes to index 0 (W-DUPC).
    {
        HandCase c;
        c.id = "H4";
        c.N = 2; c.D = 1; c.K = 2;
        c.X = {0.0f, 1.0f};
        c.C0 = {5.0f, 5.0f};
        c.expected_A = {0, 0};
        c.expected_C = {0.5f, 5.0f};
        c.expected_n = {2, 0};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.expect_warning_dupc = true;
        cases.push_back(c);
    }

    // H5: K > N, so at least K - N clusters stay empty for the whole run (VAL-003, W-KGTN).
    {
        HandCase c;
        c.id = "H5";
        c.N = 1; c.D = 1; c.K = 3;
        c.X = {4.0f};
        c.C0 = {0.0f, 1.0f, 2.0f};
        c.expected_A = {2};
        c.expected_C = {0.0f, 1.0f, 4.0f};
        c.expected_n = {0, 0, 1};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        c.expect_warning_kgtn = true;
        cases.push_back(c);
    }

    // H6: the 2-D case, which catches a transposed or stride-confused layout that every 1-D case
    // above would pass.
    {
        HandCase c;
        c.id = "H6";
        c.N = 4; c.D = 2; c.K = 2;
        c.X = {0.0f, 0.0f, 0.0f, 1.0f, 10.0f, 0.0f, 10.0f, 1.0f};
        c.C0 = {0.0f, 0.0f, 10.0f, 0.0f};
        c.expected_A = {0, 0, 1, 1};
        c.expected_C = {0.0f, 0.5f, 10.0f, 0.5f};
        c.expected_n = {2, 2};
        c.expected_iterations = 1;
        c.expected_status = Status::CONVERGED_ASSIGNMENTS;
        c.expected_centroid_consistent = true;
        cases.push_back(c);
    }

    return cases;
}

}  // namespace tests
}  // namespace kmeans

#endif  // KMEANS_TESTS_HAND_DATA_H


In [ ]:
%%writefile test_cpu.cpp
// test_cpu.cpp — CPU-level test driver for milestone M1.
//
// Covers requirements.md §16.0 (H1-H6 and H2-a..e against the CPU reference), §16.3 (the FP32 and
// FP64 distance paths agree except on near-ties), TEST-U08 (movement and the inclusive tolerance
// boundary), TEST-U09 (empty clusters), TEST-U11 (size and overflow arithmetic), TEST-U12 (file
// I/O) and TEST-U13 (generator determinism).
//
// One executable, non-zero exit on any failure, so a single notebook cell can gate the run
// (NFR-015). GPU-level versions of these cases arrive in later milestones.

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <string>
#include <vector>

#include "common.h"
#include "cpu_reference.h"
#include "gen.h"
#include "hand_data.h"
#include "io_kmd.h"
#include "validate.h"

using namespace kmeans;

namespace {

int g_checks = 0;
int g_failures = 0;
int g_case_failures = 0;

void check(bool condition, const std::string &what)
{
    ++g_checks;
    if (!condition) {
        ++g_failures;
        ++g_case_failures;
        std::printf("      FAIL  %s\n", what.c_str());
    }
}

void check_eq_i64(int64_t got, int64_t want, const std::string &what)
{
    check(got == want, what + ": got " + std::to_string(got) + ", want " + std::to_string(want));
}

std::string float_str(float v)
{
    char buf[48];
    std::snprintf(buf, sizeof(buf), "%.9g", static_cast<double>(v));
    return std::string(buf);
}

std::string double_str(double v)
{
    char buf[48];
    std::snprintf(buf, sizeof(buf), "%.17g", v);
    return std::string(buf);
}

// Scope guard for the per-test PASS/FAIL line.
class TestScope {
public:
    TestScope(const char *id, const std::string &title) : id_(id)
    {
        g_case_failures = 0;
        std::printf("  %-7s %s\n", id, title.c_str());
    }
    ~TestScope()
    {
        std::printf("      %s\n", g_case_failures == 0 ? "PASS" : "-> FAILED");
    }

private:
    const char *id_;
};

// --- §16.0 hand datasets -----------------------------------------------------------------------

void test_hand_cases()
{
    for (const tests::HandCase &c : tests::hand_cases()) {
        TestScope scope("H-case", c.id + (c.note.empty() ? "" : "  (" + c.note + ")"));

        // Validation first, because two of the cases exist to exercise VAL-003 and VAL-008.
        const Validation params = validate_parameters(c.N, c.D, c.K, c.max_iter, c.tol);
        check(params.ok, c.id + ": parameters accepted");
        double m_range = 0.0;
        const Validation data = validate_data(c.X, c.N, c.D, c.C0, c.K, &m_range);
        check(data.ok, c.id + ": data accepted");
        check(params.has_warning("W-KGTN") == c.expect_warning_kgtn,
              c.id + ": W-KGTN warning " + (c.expect_warning_kgtn ? "expected" : "not expected"));
        check(data.has_warning("W-DUPC") == c.expect_warning_dupc,
              c.id + ": W-DUPC warning " + (c.expect_warning_dupc ? "expected" : "not expected"));

        const Result r = cpu::run(c.X.data(), c.N, c.D, c.C0.data(), c.K, c.max_iter, c.tol,
                                  m_range, false);

        check_eq_i64(r.iterations, c.expected_iterations, c.id + ": iteration count");
        check(r.status == c.expected_status,
              c.id + ": status: got " + status_name(r.status) + ", want " +
                  status_name(c.expected_status));
        check(r.centroid_consistent == c.expected_centroid_consistent,
              c.id + ": P2 flag: got " + (r.centroid_consistent ? "true" : "false") + ", want " +
                  (c.expected_centroid_consistent ? "true" : "false"));
        check(r.assignment_consistent, c.id + ": P1 flag true by construction");

        for (int64_t i = 0; i < c.N; ++i) {
            check_eq_i64(r.assignments[static_cast<size_t>(i)], c.expected_A[static_cast<size_t>(i)],
                         c.id + ": A[" + std::to_string(i) + "]");
        }
        for (int64_t k = 0; k < c.K; ++k) {
            check_eq_i64(r.counts[static_cast<size_t>(k)], c.expected_n[static_cast<size_t>(k)],
                         c.id + ": n[" + std::to_string(k) + "]");
        }
        const double s_max = s_max_from_range(m_range);
        for (int64_t e = 0; e < c.K * c.D; ++e) {
            const float got = r.centroids[static_cast<size_t>(e)];
            const float want = c.expected_C[static_cast<size_t>(e)];
            const bool ok = c.centroids_exact ? (got == want) : centroid_agrees(got, want, s_max);
            check(ok, c.id + ": C[" + std::to_string(e) + "]: got " + float_str(got) + ", want " +
                          float_str(want) + (c.centroids_exact ? " (exact)" : " (NUM-006)"));
        }

        // Empty clusters must retain their initial centroid bit-exactly (DEC-003). This holds for
        // the H cases because no empty cluster is ever re-populated in them.
        for (int64_t k = 0; k < c.K; ++k) {
            if (c.expected_n[static_cast<size_t>(k)] != 0) {
                continue;
            }
            for (int64_t j = 0; j < c.D; ++j) {
                const size_t e = static_cast<size_t>(k * c.D + j);
                check(r.centroids[e] == c.C0[e],
                      c.id + ": empty cluster " + std::to_string(k) + " retained C0[" +
                          std::to_string(j) + "] bit-exactly");
            }
        }
    }
}

// --- TEST-U08: movement and the inclusive tolerance boundary -----------------------------------

void test_u08_movement()
{
    {
        TestScope scope("U08", "movement: exact delta, empty cluster excluded");
        // Cluster 0 moves (0,0) -> (3,4), a 3-4-5 triangle, so delta is exactly 5 in binary64.
        // Cluster 1 does not move. Cluster 2 is empty: its delta is 0 by construction and it is
        // excluded from the maximum, so a 0/0 division would show up here as NaN or 0.
        const std::vector<float> c_cur = {0.0f, 0.0f, 1.0f, 1.0f, 5.0f, 5.0f};
        const std::vector<float> c_next = {3.0f, 4.0f, 1.0f, 1.0f, 5.0f, 5.0f};
        const std::vector<uint32_t> n = {2, 1, 0};
        const double delta = cpu::movement(c_cur.data(), c_next.data(), n.data(), 3, 2);
        check(delta == 5.0, "delta: got " + double_str(delta) + ", want exactly 5");

        // With every cluster empty the maximum is over an empty set, which §2.6 cannot reach
        // because N >= 1, but the function must still return 0 rather than a NaN.
        const std::vector<uint32_t> none = {0, 0, 0};
        const double delta_none = cpu::movement(c_cur.data(), c_next.data(), none.data(), 3, 2);
        check(delta_none == 0.0, "delta with no non-empty cluster is 0");
    }

    {
        TestScope scope("U08", "inclusive comparison at delta == tol (DEC-002, ENV-10)");
        const std::vector<float> X = tests::h2_points();
        const std::vector<float> C0 = tests::h2_initial_centroids();
        double m_range = 0.0;
        const Validation v = validate_data(X, 4, 1, C0, 2, &m_range);
        check(v.ok, "H2 data accepted");

        // The first update of H2 moves a centroid by exactly 2. With tol = 2 the inclusive
        // comparison stops there; one ulp below 2 it must not.
        const Result at_boundary =
            cpu::run(X.data(), 4, 1, C0.data(), 2, 10, 2.0, m_range, false);
        check(at_boundary.status == Status::CONVERGED_MOVEMENT,
              std::string("tol == delta: status ") + status_name(at_boundary.status));
        check_eq_i64(at_boundary.iterations, 1, "tol == delta: iterations");

        const double just_below = std::nextafter(2.0, 0.0);
        const Result below =
            cpu::run(X.data(), 4, 1, C0.data(), 2, 10, just_below, m_range, false);
        check(below.status == Status::CONVERGED_MOVEMENT,
              std::string("tol one ulp below delta: status ") + status_name(below.status));
        check_eq_i64(below.iterations, 2, "tol one ulp below delta: iterations (not 1)");
        check(below.last_delta == 1.5, "tol one ulp below delta: last delta is the second one");
    }

    {
        TestScope scope("U08", "tol = 0 means bit-identical centroids");
        // H1 converges by assignment stability at t = 1 even though its delta (0.5) exceeds 0,
        // which is what DEC-002's "assignment stability takes precedence" means in practice.
        const std::vector<float> X = {0.0f, 1.0f, 9.0f, 10.0f};
        const std::vector<float> C0 = {0.0f, 10.0f};
        const Result r = cpu::run(X.data(), 4, 1, C0.data(), 2, 10, 0.0, 10.0, false);
        check(r.status == Status::CONVERGED_ASSIGNMENTS,
              std::string("status ") + status_name(r.status));
        check(r.last_delta == 0.5, "last delta: got " + double_str(r.last_delta) + ", want 0.5");
    }

    {
        TestScope scope("U08", "fixed-iteration mode disables both criteria (FR-020)");
        const std::vector<float> X = tests::h2_points();
        const std::vector<float> C0 = tests::h2_initial_centroids();
        const Result r = cpu::run(X.data(), 4, 1, C0.data(), 2, 7, 1e9, 10.0, true);
        check_eq_i64(r.iterations, 7, "exactly max_iter updates run");
        check(r.status == Status::ITERATION_LIMIT, std::string("status ") + status_name(r.status));
    }
}

// --- TEST-U09: empty clusters ------------------------------------------------------------------

void test_u09_empty_clusters()
{
    {
        TestScope scope("U09", "empty cluster at the update step retains its centroid");
        // The mid-iteration case of EDGE-14 at step level: cluster 1 held points in an earlier
        // accumulation and holds none now. update() must copy its centroid through untouched and
        // never divide, and movement() must report only cluster 0's displacement.
        const std::vector<float> c_cur = {1.0f, 7.25f};
        const std::vector<double> S = {11.0, 0.0};
        const std::vector<uint32_t> n = {4, 0};
        std::vector<float> c_next(2, -1.0f);
        cpu::update(S.data(), n.data(), 2, 1, c_cur.data(), c_next.data());

        check(c_next[0] == 2.75f, "non-empty cluster: 11/4 = 2.75 exactly, got " +
                                      float_str(c_next[0]));
        check(c_next[1] == 7.25f, "empty cluster retained bit-exactly, got " +
                                      float_str(c_next[1]));
        check(std::isfinite(c_next[1]), "empty cluster is not NaN or Inf (no 0/0 division)");

        const double delta = cpu::movement(c_cur.data(), c_next.data(), n.data(), 2, 1);
        check(delta == 1.75, "delta is cluster 0's 1.75 only, got " + double_str(delta));
    }

    {
        TestScope scope("U09", "single-point cluster is bit-exact (EDGE-15)");
        // Division by 1 is exact in binary64 and the quotient is representable, so the centroid
        // must equal the point bit for bit, not merely within NUM-006.
        const std::vector<float> X = {0.1f, 1234.5678f, -7.25f};
        const std::vector<float> C0 = {0.1f, 1234.5678f, -7.25f};
        const std::vector<int32_t> A = {0, 1, 2};
        std::vector<double> S(3, 0.0);
        std::vector<uint32_t> n(3, 0);
        cpu::accumulate(X.data(), 3, 1, A.data(), 3, S.data(), n.data());
        std::vector<float> C_next(3, 0.0f);
        cpu::update(S.data(), n.data(), 3, 1, C0.data(), C_next.data());
        for (int k = 0; k < 3; ++k) {
            check(C_next[static_cast<size_t>(k)] == X[static_cast<size_t>(k)],
                  "cluster " + std::to_string(k) + " equals its only point bit-exactly");
            check_eq_i64(n[static_cast<size_t>(k)], 1, "count of cluster " + std::to_string(k));
        }
    }

    {
        TestScope scope("U09", "K > N leaves at least K - N clusters empty (EDGE-04)");
        const std::vector<float> X = {4.0f};
        const std::vector<float> C0 = {0.0f, 1.0f, 2.0f};
        const Result r = cpu::run(X.data(), 1, 1, C0.data(), 3, 10, 0.0, 4.0, false);
        int empty = 0;
        for (uint32_t count : r.counts) {
            if (count == 0) {
                ++empty;
            }
        }
        check(empty >= 2, "at least K - N = 2 clusters empty, got " + std::to_string(empty));
        check(r.centroids[0] == 0.0f && r.centroids[1] == 1.0f,
              "both empty clusters retained their initial centroids");
    }
}

// --- TEST-U11: size and overflow arithmetic ----------------------------------------------------

void test_u11_size_calculator()
{
    {
        TestScope scope("U11", "MEM-002 byte counts");
        // N=1000, D=32, K=64, Approach A (G = 0), 1024 movement blocks:
        //   8ND = 256000, 4N = 4000, 16KD = 32768, 4K = 256, scalars = 1024*8 + 32 = 8224
        const MemoryModel m = device_memory_bytes(1000, 32, 64, 0, 1024);
        check(!m.overflow, "no overflow for a modest configuration");
        check_eq_i64(static_cast<int64_t>(m.x_col), 128000, "x_col bytes (4ND)");
        check_eq_i64(static_cast<int64_t>(m.x_row), 128000, "x_row bytes (4ND)");
        check_eq_i64(static_cast<int64_t>(m.centroids), 16384, "centroid bytes (8KD)");
        check_eq_i64(static_cast<int64_t>(m.assignments), 4000, "assignment bytes (4N)");
        check_eq_i64(static_cast<int64_t>(m.sums), 16384, "sum bytes (8KD)");
        check_eq_i64(static_cast<int64_t>(m.counts), 256, "count bytes (4K)");
        check_eq_i64(static_cast<int64_t>(m.partials), 0, "no P buffer for Approach A");
        check_eq_i64(static_cast<int64_t>(m.partial_counts), 0, "no Q buffer for Approach A");
        check_eq_i64(static_cast<int64_t>(m.total), 128000 + 128000 + 16384 + 4000 + 16384 + 256 +
                                                        8224,
                     "total bytes");
    }

    {
        TestScope scope("U11", "Approach B adds 8GKD + 4GK");
        const MemoryModel a = device_memory_bytes(1000, 32, 64, 0, 1024);
        const MemoryModel b = device_memory_bytes(1000, 32, 64, 7, 1024);
        check_eq_i64(static_cast<int64_t>(b.partials), 8 * 7 * 64 * 32, "P buffer bytes (8GKD)");
        check_eq_i64(static_cast<int64_t>(b.partial_counts), 4 * 7 * 64, "Q buffer bytes (4GK)");
        check_eq_i64(static_cast<int64_t>(b.total - a.total), 8 * 7 * 64 * 32 + 4 * 7 * 64,
                     "difference is exactly the P and Q buffers");
    }

    {
        TestScope scope("U11", "overflow is detected, not wrapped (CUDA-006, EDGE-30)");
        uint64_t out = 0;
        check(checked_mul(UINT64_MAX / 2, 2, &out), "a product that just fits is accepted");
        check(!checked_mul(UINT64_MAX / 2 + 1, 2, &out), "a product that overflows is rejected");
        check(!checked_add(UINT64_MAX, 1, &out), "a sum that overflows is rejected");
        check(checked_mul(0, UINT64_MAX, &out) && out == 0, "zero times anything is zero");

        const int64_t huge = 2147483647;  // 2^31 - 1, the VAL-002 maximum
        const MemoryModel m = device_memory_bytes(huge, huge, huge, huge, 1024);
        check(m.overflow, "N = D = K = G = 2^31-1 overflows the capacity model");
        check_eq_i64(static_cast<int64_t>(m.total), 0, "an overflowing model reports no total");

        const Validation v = validate_capacity(huge, huge, huge, huge, 16ull * 1024 * 1024 * 1024);
        check(!v.ok && v.exit_code == kExitInvalidInput,
              "VAL-009 rejects the overflowing configuration with exit 2");
    }

    {
        TestScope scope("U11", "VAL-009 compares against 95 % of free memory");
        // 15 GB free, as on the recorded Tesla T4 (§1.6.2).
        const uint64_t free_bytes = 15637086208ull;
        const Validation ok = validate_capacity(1000000, 32, 64, 0, free_bytes);
        check(ok.ok, "the PERF-003 base point fits");

        const MemoryModel m = device_memory_bytes(1000000, 32, 64, 0);
        const Validation tight = validate_capacity(1000000, 32, 64, 0, m.total);
        check(!tight.ok && tight.exit_code == kExitUnsupportedDevice,
              "exactly the required bytes free is rejected by the 5 % margin with exit 3");
    }

    {
        TestScope scope("U11", "VAL-001..VAL-010 reject what they must");
        check(!validate_parameters(0, 1, 1, 10, 0.0).ok, "N = 0 rejected (VAL-001, EDGE-01)");
        check(!validate_parameters(1, 0, 1, 10, 0.0).ok, "D = 0 rejected (VAL-001, EDGE-02)");
        check(!validate_parameters(1, 1, 0, 10, 0.0).ok, "K = 0 rejected (VAL-001, EDGE-03)");
        check(validate_parameters(1, 1, 1, 0, 0.0).ok, "max_iter = 0 accepted (EDGE-22)");
        check(!validate_parameters(1, 1, 1, -1, 0.0).ok, "negative max_iter rejected (VAL-004)");
        check(!validate_parameters(1, 1, 1, 10, -1e-30).ok, "negative tol rejected (VAL-005)");
        check(!validate_parameters(1, 1, 1, 10, std::nan("")).ok, "NaN tol rejected (VAL-005)");
        check(!validate_parameters(1, 1, 1, 10, HUGE_VAL).ok, "infinite tol rejected (VAL-005)");
        check(validate_parameters(2, 1, 5, 10, 0.0).has_warning("W-KGTN"),
              "K > N warns W-KGTN and continues (VAL-003)");
        check(validate_parameters(1, 1, 1, 2147483648ll, 0.0).ok == false,
              "max_iter above 2^31-1 rejected (VAL-004)");
        check(validate_parameters(2147483648ll, 1, 1, 10, 0.0).ok == false,
              "N above 2^31-1 rejected (VAL-002)");

        // VAL-006 and VAL-007 on the data.
        double m_range = 0.0;
        const std::vector<float> good = {1.0f, 2.0f};
        const std::vector<float> c_one = {1.0f};
        const std::vector<float> with_nan = {1.0f, std::nanf("")};
        const std::vector<float> with_inf = {1.0f, HUGE_VALF};
        check(validate_data(good, 2, 1, c_one, 1, &m_range).ok, "finite data accepted");
        check(m_range == 2.0, "M is the largest magnitude, got " + double_str(m_range));
        const Validation nan_v = validate_data(with_nan, 2, 1, c_one, 1, &m_range);
        check(!nan_v.ok && nan_v.rule == "VAL-006", "NaN in data rejected (EDGE-20)");
        const Validation inf_v = validate_data(with_inf, 2, 1, c_one, 1, &m_range);
        check(!inf_v.ok && inf_v.rule == "VAL-006", "Inf in data rejected (EDGE-20)");

        // VAL-007: 4 D M^2 <= 2^126. With D = 1 the largest admissible M is 2^62, and a value an
        // octave above it must be refused.
        const float at_bound = static_cast<float>(std::ldexp(1.0, 62));
        const std::vector<float> big = {at_bound};
        check(validate_data(big, 1, 1, big, 1, &m_range).ok, "M = 2^62 with D = 1 is accepted");
        const float over = static_cast<float>(std::ldexp(1.0, 63));
        const std::vector<float> too_big = {over};
        const Validation range_v = validate_data(too_big, 1, 1, too_big, 1, &m_range);
        check(!range_v.ok && range_v.rule == "VAL-007", "M = 2^63 with D = 1 is rejected");

        // VAL-008 duplicate centroids warn but do not fail.
        const std::vector<float> dup = {5.0f, 5.0f};
        const Validation dup_v = validate_data(good, 2, 1, dup, 2, &m_range);
        check(dup_v.ok && dup_v.has_warning("W-DUPC"), "duplicate centroids warn W-DUPC");

        // VAL-010 option ranges.
        check(validate_options(256, 0, 8, 32, 1024, 49152).ok, "the defaults are accepted");
        check(!validate_options(200, 0, 8, 32, 1024, 49152).ok, "block size 200 is not a multiple "
                                                                "of 32");
        check(!validate_options(2048, 0, 8, 32, 1024, 49152).ok, "block size above 1024 rejected");
        check(!validate_options(256, -1, 8, 32, 1024, 49152).ok, "negative G rejected");
        check(!validate_options(256, 0, 8, 48, 1024, 49152).ok, "tile_d not a multiple of 32");
        check(!validate_options(256, 0, 1024, 32, 1024, 49152).ok,
              "tiles larger than the shared-memory limit rejected (EDGE-34)");
    }
}

// --- TEST-U12: file I/O ------------------------------------------------------------------------

void test_u12_file_io()
{
    TestScope scope("U12", "BLD-004 .kmd/.kma round trip and rejection of damaged files");

    check(host_is_little_endian(), "host is little-endian, as BLD-004 requires");

    const std::string path = "test_u12.kmd";
    const std::vector<float> values = {0.0f, 1.5f, -2.25f, 1e30f, 10.0f, 0.1f};
    std::string err;
    check(write_kmd(path, 3, 2, values.data(), &err) == kExitOk, "write a 3x2 .kmd: " + err);

    Matrix m;
    check(read_kmd(path, &m, &err) == kExitOk, "read it back: " + err);
    check(m.rows == 3 && m.cols == 2, "dimensions survive the round trip");
    check(m.data == values, "every value survives the round trip bit-exactly");

    // The payload of this file contains byte 0x0A, which Windows text mode would translate. A
    // failure here means a stream was opened without std::ios::binary.
    const std::vector<float> newline_byte = {1.4012985e-45f * 10.0f};  // 0x0000000A as a float
    const float raw_newline = []() {
        uint32_t bits = 0x0000000Au;
        float f = 0.0f;
        std::memcpy(&f, &bits, sizeof(f));
        return f;
    }();
    const std::vector<float> with_lf = {raw_newline, raw_newline, 1.0f};
    check(write_kmd("test_u12_lf.kmd", 3, 1, with_lf.data(), &err) == kExitOk,
          "write a payload containing 0x0A bytes");
    Matrix lf;
    check(read_kmd("test_u12_lf.kmd", &lf, &err) == kExitOk, "read it back: " + err);
    check(lf.data == with_lf, "0x0A payload bytes are not translated (binary mode)");
    (void)newline_byte;

    // FR-002: the declared shape must match the header.
    check(read_kmd_expect(path, 3, 2, &m, &err) == kExitOk, "matching declared shape accepted");
    const int mismatch = read_kmd_expect(path, 4, 2, &m, &err);
    check(mismatch == kExitInvalidInput, "declared shape mismatch is exit 2, got " +
                                             std::to_string(mismatch));

    // Truncation and trailing bytes are both caught by the exact-size rule.
    {
        std::vector<unsigned char> bytes;
        std::FILE *f = std::fopen(path.c_str(), "rb");
        check(f != nullptr, "reopen the file to copy its bytes");
        if (f != nullptr) {
            unsigned char buf[512];
            const size_t got = std::fread(buf, 1, sizeof(buf), f);
            bytes.assign(buf, buf + got);
            std::fclose(f);
        }

        std::FILE *t = std::fopen("test_u12_trunc.kmd", "wb");
        if (t != nullptr && bytes.size() > 4) {
            std::fwrite(bytes.data(), 1, bytes.size() - 4, t);
            std::fclose(t);
        }
        const int code = read_kmd("test_u12_trunc.kmd", &m, &err);
        check(code == kExitIoError, "truncated file is exit 5, got " + std::to_string(code));

        std::FILE *x = std::fopen("test_u12_extra.kmd", "wb");
        if (x != nullptr) {
            std::fwrite(bytes.data(), 1, bytes.size(), x);
            const unsigned char junk[4] = {1, 2, 3, 4};
            std::fwrite(junk, 1, sizeof(junk), x);
            std::fclose(x);
        }
        const int extra = read_kmd("test_u12_extra.kmd", &m, &err);
        check(extra == kExitIoError, "trailing bytes give exit 5, got " + std::to_string(extra));

        std::FILE *b = std::fopen("test_u12_magic.kmd", "wb");
        if (b != nullptr && bytes.size() > 4) {
            std::fwrite("XXXX", 1, 4, b);
            std::fwrite(bytes.data() + 4, 1, bytes.size() - 4, b);
            std::fclose(b);
        }
        const int magic = read_kmd("test_u12_magic.kmd", &m, &err);
        check(magic == kExitIoError, "bad magic gives exit 5, got " + std::to_string(magic));
    }

    const int missing = read_kmd("test_u12_does_not_exist.kmd", &m, &err);
    check(missing == kExitIoError, "missing file is exit 5, got " + std::to_string(missing));

    // .kma assignments.
    const std::vector<int32_t> assignments = {0, 1, 2, 1, 0, 2147483647};
    check(write_kma("test_u12.kma", assignments, &err) == kExitOk, "write .kma: " + err);
    std::vector<int32_t> read_back;
    check(read_kma("test_u12.kma", &read_back, &err) == kExitOk, "read .kma: " + err);
    check(read_back == assignments, ".kma round trip is bit-exact");

    for (const char *p : {"test_u12.kmd", "test_u12_lf.kmd", "test_u12_trunc.kmd",
                          "test_u12_extra.kmd", "test_u12_magic.kmd", "test_u12.kma"}) {
        std::remove(p);
    }
}

// --- TEST-U13: generator determinism -----------------------------------------------------------

void test_u13_generator()
{
    {
        TestScope scope("U13", "identical parameters reproduce identical datasets (FR-003)");
        GenParams p;
        p.N = 500;
        p.D = 4;
        p.K = 5;
        p.sep = 10.0;
        p.spread = 1.0;
        p.seed = 20261017;

        const Dataset a = generate(p);
        const Dataset b = generate(p);
        check(a.X == b.X, "the data matrix is bit-identical across runs");
        check(a.C0 == b.C0, "the initial centroids are bit-identical across runs");
        check(a.dataset_id == b.dataset_id, "the dataset ID is stable: " + a.dataset_id);
        check_eq_i64(static_cast<int64_t>(a.X.size()), p.N * p.D, "data size is N*D");
        check_eq_i64(static_cast<int64_t>(a.C0.size()), p.K * p.D, "centroid size is K*D");

        // A different seed must change the data; otherwise the seed is being ignored.
        GenParams q = p;
        q.seed = p.seed + 1;
        const Dataset c = generate(q);
        check(c.X != a.X, "a different seed changes the data");
        check(c.dataset_id != a.dataset_id, "a different seed changes the dataset ID");

        // The written files must be byte-identical too, which is what TEST-U13 compares.
        std::string err;
        check(write_kmd("test_u13_a.kmd", static_cast<uint64_t>(p.N),
                        static_cast<uint64_t>(p.D), a.X.data(), &err) == kExitOk, "write A");
        check(write_kmd("test_u13_b.kmd", static_cast<uint64_t>(p.N),
                        static_cast<uint64_t>(p.D), b.X.data(), &err) == kExitOk, "write B");
        Matrix ma, mb;
        check(read_kmd("test_u13_a.kmd", &ma, &err) == kExitOk, "read A");
        check(read_kmd("test_u13_b.kmd", &mb, &err) == kExitOk, "read B");
        check(ma.data == mb.data, "the two files hold identical bytes");
        std::remove("test_u13_a.kmd");
        std::remove("test_u13_b.kmd");

        // Every generated value must pass the VAL rules the loader will apply.
        double m_range = 0.0;
        const Validation v = validate_data(a.X, p.N, p.D, a.C0, p.K, &m_range);
        check(v.ok, "generated data passes VAL-006 and VAL-007");
        check(a.min_centre_distance > 0.0, "a positive minimum centre distance is measured");
    }

    {
        TestScope scope("U13", "well-separated blobs are separated by at least 10 sigma (§10.3)");
        GenParams p;
        p.N = 1000;
        p.D = 8;
        p.K = 4;
        p.sep = 10.0;
        p.spread = 1.0;
        p.seed = 7;
        const Dataset d = generate(p);
        const double sigma = d.min_centre_distance / p.spread;
        check(sigma >= 10.0, "minimum centre separation is " + double_str(sigma) +
                                 " sigma, want >= 10 for the well-separated datasets");
    }

    {
        TestScope scope("U13", "the uniform draw stays inside (0, 1)");
        // log(u1) in Box-Muller must never see 0, and a draw of exactly 1 would bias the angle.
        Rng rng(12345);
        double lo = 1.0;
        double hi = 0.0;
        for (int i = 0; i < 200000; ++i) {
            const double u = rng.uniform_open();
            if (u < lo) {
                lo = u;
            }
            if (u > hi) {
                hi = u;
            }
        }
        check(lo > 0.0, "smallest draw is above 0: " + double_str(lo));
        check(hi < 1.0, "largest draw is below 1: " + double_str(hi));
    }

    {
        TestScope scope("U13", "the normal draw has the expected first two moments");
        // A sanity check on Box-Muller, not a statistical test: with 400k samples the sample mean
        // is within about 0.005 of 0 and the variance within about 0.01 of 1.
        Rng rng(999);
        const int count = 400000;
        double sum = 0.0;
        double sum_sq = 0.0;
        for (int i = 0; i < count; ++i) {
            const double z = rng.normal();
            sum += z;
            sum_sq += z * z;
        }
        const double mean = sum / count;
        const double variance = sum_sq / count - mean * mean;
        check(std::fabs(mean) < 0.02, "sample mean " + double_str(mean) + " is near 0");
        check(std::fabs(variance - 1.0) < 0.02,
              "sample variance " + double_str(variance) + " is near 1");
    }
}

// --- §16.3: the two distance paths agree except on near-ties -----------------------------------

void test_distance_paths_agree()
{
    TestScope scope("U01-pre", "FP32 and FP64 assignment agree except on near-ties (NUM-005)");

    GenParams p;
    p.N = 2000;
    p.D = 6;
    p.K = 8;
    p.sep = 3.0;   // deliberately tight, so near-ties actually occur
    p.spread = 1.0;
    p.seed = 424242;
    const Dataset d = generate(p);

    int64_t mismatches = 0;
    int64_t near_ties = 0;
    int64_t hard_mismatches = 0;
    double worst_margin = 0.0;

    for (int64_t i = 0; i < p.N; ++i) {
        const float *x = d.X.data() + i * p.D;
        const int32_t k32 = cpu::nearest_centroid(x, p.D, d.C0.data(), p.K);
        const cpu::TieInfo info = cpu::tie_info(x, p.D, d.C0.data(), p.K);
        if (info.near_tie) {
            ++near_ties;
        }
        if (k32 != info.k1) {
            ++mismatches;
            if (!info.near_tie) {
                ++hard_mismatches;
                if (info.margin > worst_margin) {
                    worst_margin = info.margin;
                }
            }
        }
    }

    std::printf("      %lld points, %lld near-ties, %lld FP32/FP64 mismatches, "
                "%lld of them outside the near-tie band\n",
                static_cast<long long>(p.N), static_cast<long long>(near_ties),
                static_cast<long long>(mismatches), static_cast<long long>(hard_mismatches));
    check(hard_mismatches == 0,
          "no mismatch outside the near-tie band; worst offending margin " +
              double_str(worst_margin) + " against a threshold of " +
              double_str(near_tie_threshold(p.D)));

    // Random blobs rarely land on a decision boundary, so the band above is usually empty and the
    // check passes without exercising NUM-005 at all. These constructed points sit exactly on, or
    // one ulp off, the perpendicular bisector of two centroids, which is where the rule bites
    // (EDGE-13, EDGE-19; TEST-U02 repeats them on the device).
    const std::vector<float> C = {0.0f, 0.0f, 2.0f, 0.0f};  // two centroids 2 apart in 2-D
    const std::vector<float> on_bisector = {1.0f, 0.0f, 1.0f, 5.0f, 1.0f, -3.25f};
    for (int64_t i = 0; i < 3; ++i) {
        const float *x = on_bisector.data() + i * 2;
        const int32_t k = cpu::nearest_centroid(x, 2, C.data(), 2);
        const cpu::TieInfo info = cpu::tie_info(x, 2, C.data(), 2);
        check_eq_i64(k, 0, "point on the bisector takes the lowest index (DEC-005)");
        check(info.margin == 0.0, "its binary64 margin is exactly 0");
        check(info.near_tie, "it is classified as a near-tie");
    }

    // Just off the bisector the nearer centroid must win outright, and the margin must be small
    // enough to still count as a near-tie, which is what makes a device mismatch here reportable
    // rather than a failure.
    const float nudge = std::nextafter(1.0f, 2.0f);
    const std::vector<float> off_bisector = {nudge, 0.0f};
    const cpu::TieInfo off = cpu::tie_info(off_bisector.data(), 2, C.data(), 2);
    check_eq_i64(cpu::nearest_centroid(off_bisector.data(), 2, C.data(), 2), 1,
                 "one ulp past the bisector the second centroid wins");
    check(off.near_tie, "a one-ulp margin is inside the near-tie band: margin " +
                            double_str(off.margin) + " <= " + double_str(near_tie_threshold(2)));

    // A point far from the boundary must not be excused as a near-tie.
    const std::vector<float> clear = {0.25f, 0.0f};
    const cpu::TieInfo far = cpu::tie_info(clear.data(), 2, C.data(), 2);
    check(!far.near_tie, "a clearly nearer centroid is not a near-tie: margin " +
                             double_str(far.margin));
    check_eq_i64(far.k1, 0, "and it is the lower-index one here");

    // Duplicate centroids make every point an exact tie (EDGE-10, EDGE-12).
    const std::vector<float> dup = {3.0f, 3.0f, 3.0f, 3.0f};
    const std::vector<float> any_point = {-7.5f, 2.0f};
    check_eq_i64(cpu::nearest_centroid(any_point.data(), 2, dup.data(), 2), 0,
                 "with duplicate centroids the lower index always wins");
}

// --- end-to-end sanity on generated data -------------------------------------------------------

void test_end_to_end_blobs()
{
    TestScope scope("I11-pre", "well-separated blobs recover their structure on the CPU");

    GenParams p;
    p.N = 600;
    p.D = 3;
    p.K = 4;
    p.sep = 20.0;
    p.spread = 1.0;
    p.seed = 31337;
    const Dataset d = generate(p);

    double m_range = 0.0;
    const Validation v = validate_data(d.X, p.N, p.D, d.C0, p.K, &m_range);
    check(v.ok, "generated data is valid");

    const Result r = cpu::run(d.X.data(), p.N, p.D, d.C0.data(), p.K, 100, 1e-6, m_range, false);
    check(is_converged(r.status), std::string("the run converged: ") + status_name(r.status));
    check(r.centroid_consistent, "P2 holds after convergence by assignment stability");
    check(r.assignment_consistent, "P1 holds by construction");

    uint64_t total = 0;
    for (uint32_t count : r.counts) {
        total += count;
    }
    check_eq_i64(static_cast<int64_t>(total), p.N, "the counts sum to N");

    for (int64_t i = 0; i < p.N; ++i) {
        if (r.assignments[static_cast<size_t>(i)] < 0 ||
            r.assignments[static_cast<size_t>(i)] >= p.K) {
            check(false, "assignment " + std::to_string(i) + " is outside [0, K)");
            break;
        }
    }

    // P1 independently: every reported assignment is the nearest centroid of the reported
    // centroids (§2.8 verification mode, TEST-I07 at CPU level).
    int64_t p1_violations = 0;
    for (int64_t i = 0; i < p.N; ++i) {
        const int32_t expect =
            cpu::nearest_centroid(d.X.data() + i * p.D, p.D, r.centroids.data(), p.K);
        if (expect != r.assignments[static_cast<size_t>(i)]) {
            ++p1_violations;
        }
    }
    check_eq_i64(p1_violations, 0, "P1 re-checked against the reported centroids");
    check(r.objective > 0.0, "the objective J is positive and finite");
    check(std::isfinite(r.objective), "the objective J is finite");
}

}  // namespace

int main()
{
    std::printf("M1 CPU test suite (requirements.md §16.0, §16.3, TEST-U08/U09/U11/U12/U13)\n\n");

    test_hand_cases();
    std::printf("\n");
    test_u08_movement();
    test_u09_empty_clusters();
    test_u11_size_calculator();
    test_u12_file_io();
    test_u13_generator();
    test_distance_paths_agree();
    test_end_to_end_blobs();

    std::printf("\n%d checks, %d failures\n", g_checks, g_failures);
    if (g_failures != 0) {
        std::printf("RESULT: FAILED\n");
        return 1;
    }
    std::printf("RESULT: all CPU-level tests passed\n");
    return 0;
}


## 6. Compile the host code

Builds the two host executables with the BLD-002 host flags, extended with `-Wall -Wextra`:

```
-std=c++17 -O2 -ffp-contract=off -Wall -Wextra
```

`-ffp-contract=off` (NUM-003) is the one flag that changes results rather than diagnostics. The
CPU reference is the oracle the device is judged against, and §2.2 fixes the distance as a
binary32 multiply followed by a binary32 add. If the compiler were allowed to contract that pair
into an FMA, the oracle's rounding would shift, every near-tie margin computed from it would be
measured against the wrong baseline, and NUM-005 mismatch diagnosis would become meaningless. The
device is permitted to contract (`-fmad=true`), which is precisely why the reference must not.

The cell treats **any** compiler output as a failure, not only a non-zero exit status: NFR-014
requires the compile cells to produce no host-code warnings, and a warning that merely scrolls past
in a saved notebook is indistinguishable from one nobody read.

In [ ]:
# Host compile cell — BLD-002 host flags plus -Wall -Wextra; no warnings tolerated (NFR-014).
import subprocess

if "GXX_REFERENCE_FLAGS" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

# One source of truth: section 3 owns -O2 -std=c++17 -ffp-contract=off (NUM-003, BLD-002).
GXX_FLAGS = GXX_REFERENCE_FLAGS + ["-Wall", "-Wextra"]
CPU_LIB_SOURCES = ["validate.cpp", "io_kmd.cpp", "cpu_reference.cpp", "gen.cpp"]


def compile_host(output, sources):
    """Compile one host executable; raise on a non-zero exit or on any compiler output."""
    cmd = ["g++"] + GXX_FLAGS + ["-I", "."] + sources + ["-o", output]
    print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    diagnostics = (proc.stdout + proc.stderr).strip()
    if diagnostics:
        print(diagnostics)
    if proc.returncode != 0:
        raise RuntimeError(f"g++ failed with exit code {proc.returncode} building {output}")
    if diagnostics:
        raise RuntimeError(
            f"g++ emitted diagnostics while building {output}; NFR-014 requires a warning-free "
            "host build. Fix the source in src/ or tests/, regenerate the notebook, and re-run."
        )
    print(f"  -> {output}")


compile_host("kmeans-gen", ["gen_main.cpp", "gen.cpp", "io_kmd.cpp", "validate.cpp"])
compile_host("test_cpu", ["test_cpu.cpp"] + CPU_LIB_SOURCES)
print("\nhost build clean: kmeans-gen and test_cpu, no warnings")

## 7. Hand datasets H1–H6 and the CPU-level tests

Runs the test executable, which checks the CPU reference against every hand-verifiable case of
§16.0 — H1–H6 and the five H2 variants — and then TEST-U08, U09, U11, U12 and U13 at host level.
The cell raises if any check fails, so a broken oracle stops "restart kernel and run all" here,
before any GPU result could be compared against it (NFR-015).

What the hand cases pin down, case by case:

| Case | What it fixes |
|---|---|
| H1 | the ordinary path: one update, then assignment stability, `t = 1` |
| H2 | a three-update trajectory with the deltas 2, 1.5, 3.5, ending in `CONVERGED_ASSIGNMENTS` |
| H2-a, H2-b | $\Delta \le \varepsilon$ **inclusive** (DEC-002, ENV-10): the tolerance equals the delta exactly, and the run must stop |
| H2-c | one notch below that delta, so the run continues — the other side of the same boundary |
| H2-d, H2-e | the iteration limit, including $I_{\max} = 0$: `ITERATION_LIMIT`, never reported as convergence (§2.7) |
| H3 | an exact tie resolved to the lowest index (DEC-005) and an empty cluster retained (DEC-003) |
| H4 | duplicate initial centroids (`W-DUPC`): every point ties, so index 0 takes them all |
| H5 | $K > N$ (`W-KGTN`): at least $K - N$ clusters stay empty for the whole run |
| H6 | the only 2-D case, which catches a transposed or stride-confused layout that every 1-D case above would pass |

The P2 flag is checked in every case, including the four where it is correctly **false**: after
`CONVERGED_MOVEMENT` or `ITERATION_LIMIT` the reported centroids are generally not the means of
the reported assignments, and §2.8 requires that to be reported honestly rather than repaired by
extra hidden iterations.

The cell finishes by generating one small Gaussian-blob dataset through the `kmeans-gen` CLI
(FR-003, BLD-003a), which confirms the binary format round-trips in this environment and prints
the measured minimum centre separation in units of the spread — the quantity §10.3 uses to call a
dataset well-separated.

In [ ]:
# CPU-level test suite: §16.0 hand datasets plus TEST-U08/U09/U11/U12/U13. Raises on any failure.
import os
import subprocess

if "DATA_DIR" not in globals():
    raise RuntimeError("Run the configuration cell (section 3) first.")

tests = subprocess.run(["./test_cpu"], capture_output=True, text=True)
print(tests.stdout + tests.stderr, end="")
if tests.returncode != 0:
    raise RuntimeError(
        f"./test_cpu exited with code {tests.returncode}: the CPU reference does not reproduce the "
        "hand-verified results of requirements.md §16.0. Every GPU comparison depends on this "
        "oracle, so fix it in src/ before continuing."
    )

# FR-003 / BLD-003a: one small dataset, to confirm the generator and the BLD-004 format work here.
prefix = os.path.join(DATA_DIR, "demo_n2000_d4_k3")
gen = subprocess.run(
    ["./kmeans-gen", "--n", "2000", "--d", "4", "--k", "3", "--sep", "10", "--spread", "1",
     "--seed", "20261017", "--out-prefix", prefix],
    capture_output=True, text=True,
)
print("\n$ kmeans-gen --n 2000 --d 4 --k 3 --sep 10 --spread 1 --seed 20261017")
print(gen.stdout + gen.stderr, end="")
if gen.returncode != 0:
    raise RuntimeError(f"kmeans-gen exited with code {gen.returncode} (see BLD-005 for the codes)")

for suffix, expected_rows in (("_X.kmd", 2000), ("_C0.kmd", 3)):
    path = prefix + suffix
    # BLD-004: 24-byte header (magic, version, rows, cols) then rows*cols binary32.
    expected_bytes = 24 + expected_rows * 4 * 4
    actual = os.path.getsize(path)
    if actual != expected_bytes:
        raise RuntimeError(f"{path} is {actual} B, expected {expected_bytes} B for "
                           f"{expected_rows}x4 binary32 values plus the BLD-004 header")
    print(f"  {os.path.basename(path)}: {actual} B as the format requires")